# 模块概述

WtPorter 是 WonderTrader 框架的 C 接口导出模块，用于外部语言（如 Python、C#、Java 等）与 WonderTrader C++ 核心引擎进行交互。该模块通过 C 接口和回调函数机制，实现了跨语言调用和事件驱动的编程模型。

主要包括：
- C 接口导出层：提供统一的 C 风格函数接口，供外部语言调用
- 运行时运行器：管理整个交易系统的运行时环境
- 扩展策略上下文：适配器类，将引擎事件转发给外部语言回调
- 扩展组件：支持外部语言实现的 Parser 和 Executer
- 数据加载支持：支持外部数据源加载历史数据
- 回调函数管理：统一管理各种策略和组件的事件回调

1. **C接口导出层**（WtPorter.h/cpp）：
   - 提供 C 风格导出函数，供外部语言调用
   - 包括策略回调注册、引擎初始化配置、策略上下文创建、交易操作、数据查询等接口
   - 使用单例模式管理 WtRtRunner 对象
   - 通过上下文句柄（CtxHandler）管理策略实例，避免直接暴露 C++ 对象指针

2. **运行时运行器层**（WtRtRunner）：
   - WtRtRunner：运行时运行器核心，管理整个交易系统的生命周期
   - 管理交易引擎（CTA、HFT、SEL引擎）的创建和运行
   - 管理策略上下文的创建和映射
   - 管理回调函数的注册和事件转发
   - 支持外部数据加载器和扩展组件管理

3. **扩展策略上下文层**（ExpCtaContext + ExpHftContext + ExpSelContext）：
   - ExpCtaContext：CTA策略扩展上下文，继承自CtaStraBaseCtx，转发CTA策略事件
   - ExpHftContext：HFT策略扩展上下文，继承自HftStraBaseCtx，转发HFT策略事件
   - ExpSelContext：SEL策略扩展上下文，继承自SelStraBaseCtx，转发SEL策略事件
   - 作为适配器，将策略的各种事件（初始化、交易日事件、Tick更新、K线闭合等）转发给外部语言

4. **扩展组件层**（ExpParser + ExpExecuter）：
   - ExpParser：扩展行情解析器，继承自IParserApi，允许外部语言实现Parser
   - ExpExecuter：扩展执行器，继承自IExecCommand，允许外部语言实现Executer
   - 通过回调函数机制实现外部语言与框架的交互

5. **定义层**（PorterDefs.h）：
   - 定义回调函数类型、事件常量、上下文句柄类型等基础定义
   - 提供统一的类型定义，供C接口和外部语言使用

6. **核心引擎层**（依赖WtCore模块）：
   - WtCtaEngine：CTA策略引擎
   - WtHftEngine：HFT策略引擎
   - WtSelEngine：SEL策略引擎
   - TraderAdapterMgr：交易适配器管理器
   - ParserAdapterMgr：行情解析器适配器管理器
   - WtDtMgr：数据管理器

# 层次关系图
```mermaid
graph TB
    %% 样式定义
    classDef cInterfaceClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef runnerClass fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef contextClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef adapterClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef engineClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef componentClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef baseClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% C接口导出层
    subgraph CInterfaceLayer["C接口导出层 - 外部语言绑定"]
        direction TB
        WtPorter["WtPorter.h/cpp<br/>C接口导出<br/>• 策略回调注册<br/>• 引擎初始化配置<br/>• 策略上下文创建<br/>• 交易操作接口<br/>• 数据查询接口<br/>• 扩展组件管理"]:::cInterfaceClass
    end

    %% 运行时运行器层
    subgraph RunnerLayer["运行时运行器层 - 核心调度"]
        direction TB
        WtRtRunner["WtRtRunner<br/>运行时运行器<br/>• 引擎生命周期管理<br/>• 策略上下文管理<br/>• 回调函数管理<br/>• 数据加载管理<br/>• 扩展组件管理<br/>• 事件转发"]:::runnerClass
    end

    %% 扩展策略上下文层
    subgraph ContextLayer["扩展策略上下文层 - 事件适配"]
        direction TB
        ExpCtaContext["ExpCtaContext<br/>CTA策略扩展上下文<br/>• 继承CtaStraBaseCtx<br/>• 事件转发适配<br/>• 初始化/Tick/Bar/Calc事件"]:::contextClass
        ExpHftContext["ExpHftContext<br/>HFT策略扩展上下文<br/>• 继承HftStraBaseCtx<br/>• 事件转发适配<br/>• 订单/成交/Level2事件"]:::contextClass
        ExpSelContext["ExpSelContext<br/>SEL策略扩展上下文<br/>• 继承SelStraBaseCtx<br/>• 事件转发适配<br/>• 调度/Tick/Bar事件"]:::contextClass
    end

    %% 扩展组件层
    subgraph ComponentLayer["扩展组件层 - 外部组件支持"]
        direction TB
        ExpParser["ExpParser<br/>扩展行情解析器<br/>• 继承IParserApi<br/>• 外部Parser适配<br/>• 事件/订阅回调"]:::componentClass
        ExpExecuter["ExpExecuter<br/>扩展执行器<br/>• 继承IExecCommand<br/>• 外部Executer适配<br/>• 初始化/命令回调"]:::componentClass
    end

    %% 定义层
    subgraph DefLayer["定义层 - 基础类型"]
        direction TB
        PorterDefs["PorterDefs.h<br/>Porter模块定义<br/>• 回调函数类型<br/>• 事件常量<br/>• 上下文句柄类型<br/>• 数据类型定义"]:::defClass
    end

    %% 核心引擎层（依赖）
    subgraph CoreEngineLayer["核心引擎层 - 依赖WtCore模块"]
        direction TB
        WtCtaEngine["WtCtaEngine<br/>CTA策略引擎"]:::engineClass
        WtHftEngine["WtHftEngine<br/>HFT策略引擎"]:::engineClass
        WtSelEngine["WtSelEngine<br/>SEL策略引擎"]:::engineClass
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器"]:::engineClass
        ParserAdapterMgr["ParserAdapterMgr<br/>行情适配器管理器"]:::engineClass
        WtDtMgr["WtDtMgr<br/>数据管理器"]:::engineClass
    end

    %% 基础上下文类（引用）
    subgraph BaseContextLayer["基础上下文类 - 来自WtCore模块"]
        direction TB
        CtaStraBaseCtx["CtaStraBaseCtx<br/>CTA策略基础上下文"]:::baseClass
        HftStraBaseCtx["HftStraBaseCtx<br/>HFT策略基础上下文"]:::baseClass
        SelStraBaseCtx["SelStraBaseCtx<br/>SEL策略基础上下文"]:::baseClass
    end

    %% 接口类（引用）
    subgraph InterfaceLayer["接口类 - 来自Includes模块"]
        direction TB
        IParserApi["IParserApi<br/>行情解析器API接口"]:::baseClass
        IExecCommand["IExecCommand<br/>执行器命令接口"]:::baseClass
    end

    %% C接口到运行器的关系
    WtPorter -->|"调用"| WtRtRunner
    
    %% 运行器到引擎的关系
    WtRtRunner -->|"管理"| WtCtaEngine
    WtRtRunner -->|"管理"| WtHftEngine
    WtRtRunner -->|"管理"| WtSelEngine
    WtRtRunner -->|"管理"| TraderAdapterMgr
    WtRtRunner -->|"管理"| ParserAdapterMgr
    WtRtRunner -->|"管理"| WtDtMgr
    
    %% 运行器到扩展上下文的关系
    WtRtRunner -->|"创建"| ExpCtaContext
    WtRtRunner -->|"创建"| ExpHftContext
    WtRtRunner -->|"创建"| ExpSelContext
    WtRtRunner -->|"创建"| ExpParser
    WtRtRunner -->|"创建"| ExpExecuter
    
    %% 扩展上下文继承关系
    ExpCtaContext -.->|"继承"| CtaStraBaseCtx
    ExpHftContext -.->|"继承"| HftStraBaseCtx
    ExpSelContext -.->|"继承"| SelStraBaseCtx
    
    %% 扩展组件继承关系
    ExpParser -.->|"实现"| IParserApi
    ExpExecuter -.->|"实现"| IExecCommand
    
    %% 引擎到基础上下文的关系
    WtCtaEngine -->|"创建"| CtaStraBaseCtx
    WtHftEngine -->|"创建"| HftStraBaseCtx
    WtSelEngine -->|"创建"| SelStraBaseCtx
    
    %% 扩展上下文到引擎的关系
    ExpCtaContext -->|"使用"| WtCtaEngine
    ExpHftContext -->|"使用"| WtHftEngine
    ExpSelContext -->|"使用"| WtSelEngine
    
    %% 回调函数流
    WtRtRunner -.->|"注册回调"| PorterDefs
    ExpCtaContext -.->|"事件回调"| WtRtRunner
    ExpHftContext -.->|"事件回调"| WtRtRunner
    ExpSelContext -.->|"事件回调"| WtRtRunner
    ExpParser -.->|"事件回调"| WtRtRunner
    ExpExecuter -.->|"事件回调"| WtRtRunner
    
    %% 外部语言交互
    ExternalLang["外部语言<br/>(Python/C#等)"] -.->|"调用C接口"| WtPorter
    ExternalLang -.->|"注册回调"| WtPorter
    WtPorter -.->|"事件通知"| ExternalLang
    
    %% 应用样式
    class WtPorter cInterfaceClass
    class WtRtRunner runnerClass
    class ExpCtaContext,ExpHftContext,ExpSelContext contextClass
    class ExpParser,ExpExecuter componentClass
    class WtCtaEngine,WtHftEngine,WtSelEngine,TraderAdapterMgr,ParserAdapterMgr,WtDtMgr engineClass
    class PorterDefs defClass
    class CtaStraBaseCtx,HftStraBaseCtx,SelStraBaseCtx,IParserApi,IExecCommand baseClass
```

# 基础类型定义 PorterDefs.h

# C 接口导出 WtPorter.h/cpp

## 获取WtRtRunner单例对象 getRunner
```cpp
/**
 * @brief 获取WtRtRunner单例对象
 * 
 * 使用静态局部变量实现单例模式，确保全局只有一个WtRtRunner实例
 * 
 * @return WtRtRunner对象的引用
 */
WtRtRunner& getRunner()
{
	static WtRtRunner runner;
	return runner;
}
```

## 回调函数注册接口

### 注册引擎事件回调函数 register_evt_callback
```cpp
/**
 * @brief 注册引擎事件回调函数
 * 
 * 将引擎事件回调函数注册到WtRtRunner中
 * 
 * @param cbEvt 事件回调函数指针
 */
void register_evt_callback(FuncEventCallback cbEvt)
{
	getRunner().registerEvtCallback(cbEvt);
}
```

### 注册CTA策略回调函数 register_cta_callbacks
```cpp
/**
 * @brief 注册CTA策略回调函数
 * 
 * 将CTA策略的所有回调函数注册到WtRtRunner中
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbCalc 策略计算回调函数
 * @param cbBar K线闭合回调函数
 * @param cbSessEvt 交易日事件回调函数
 * @param cbCondTrigger 条件单触发回调函数（可选）
 */
void register_cta_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt, FuncStraCondTriggerCallback cbCondTrigger/* = NULL*/)
{
	getRunner().registerCtaCallbacks(cbInit, cbTick, cbCalc, cbBar, cbSessEvt, cbCondTrigger);
}
```

### 注册选股策略回调函数 register_sel_callbacks
```cpp
/**
 * @brief 注册选股策略回调函数
 * 
 * 将SEL策略的所有回调函数注册到WtRtRunner中
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbCalc 策略计算回调函数
 * @param cbBar K线闭合回调函数
 * @param cbSessEvt 交易日事件回调函数
 */
void register_sel_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt)
{
	getRunner().registerSelCallbacks(cbInit, cbTick, cbCalc, cbBar, cbSessEvt);
}
```

### 注册HFT策略回调函数 register_hft_callbacks
```cpp
/**
 * @brief 注册HFT策略回调函数
 * 
 * 将HFT策略的所有回调函数注册到WtRtRunner中
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbBar K线闭合回调函数
 * @param cbChnl 交易通道事件回调函数
 * @param cbOrd 订单状态回调函数
 * @param cbTrd 成交回报回调函数
 * @param cbEntrust 委托回报回调函数
 * @param cbOrdDtl 订单明细回调函数
 * @param cbOrdQue 订单队列回调函数
 * @param cbTrans 逐笔成交回调函数
 * @param cbSessEvt 交易日事件回调函数
 * @param cbPosition 持仓变化回调函数
 */
void register_hft_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraBarCallback cbBar, 
	FuncHftChannelCallback cbChnl, FuncHftOrdCallback cbOrd, FuncHftTrdCallback cbTrd, FuncHftEntrustCallback cbEntrust,
	FuncStraOrdDtlCallback cbOrdDtl, FuncStraOrdQueCallback cbOrdQue, FuncStraTransCallback cbTrans, FuncSessionEvtCallback cbSessEvt, FuncHftPosCallback cbPosition)
{
	getRunner().registerHftCallbacks(cbInit, cbTick, cbBar, cbChnl, cbOrd, cbTrd, cbEntrust, cbOrdDtl, cbOrdQue, cbTrans, cbSessEvt, cbPosition);
}
```

### 注册扩展Parser回调函数 register_parser_callbacks
```cpp
/**
 * @brief 注册扩展Parser回调函数
 * 
 * 将扩展Parser的事件和订阅回调函数注册到WtRtRunner中
 * 
 * @param cbEvt Parser事件回调函数
 * @param cbSub Parser订阅回调函数
 */
void register_parser_callbacks(FuncParserEvtCallback cbEvt, FuncParserSubCallback cbSub)
{
	getRunner().registerParserPorter(cbEvt, cbSub);
}
```

### 注册扩展Executer回调函数 register_exec_callbacks
```cpp
/**
 * @brief 注册扩展Executer回调函数
 * 
 * 将扩展Executer的初始化和命令回调函数注册到WtRtRunner中
 * 
 * @param cbInit 执行器初始化回调函数
 * @param cbExec 执行器命令回调函数
 */
void register_exec_callbacks(FuncExecInitCallback cbInit, FuncExecCmdCallback cbExec)
{
	getRunner().registerExecuterPorter(cbInit, cbExec);
}
```

### 注册外部数据加载器 register_ext_data_loader
```cpp
/**
 * @brief 注册外部数据加载器
 * 
 * 将外部数据加载器的回调函数注册到WtRtRunner中
 * 
 * @param fnlBarLoader 加载复权K线数据的回调函数
 * @param rawBarLoader 加载原始K线数据的回调函数
 * @param fctLoader 加载复权因子的回调函数
 * @param tickLoader 加载Tick数据的回调函数
 */
void register_ext_data_loader(FuncLoadFnlBars fnlBarLoader, FuncLoadRawBars rawBarLoader, FuncLoadAdjFactors fctLoader, FuncLoadRawTicks tickLoader)
{
	getRunner().registerExtDataLoader(fnlBarLoader, rawBarLoader, fctLoader, tickLoader);
}
```

## 数据推送接口

### 推送原始K线数据 feed_raw_bars
```cpp
/**
 * @brief 推送原始K线数据
 * 
 * 将外部数据源的原始K线数据推送到WtRtRunner中
 * 
 * @param bars K线数据数组指针
 * @param count K线数据条数
 */
void feed_raw_bars(WTSBarStruct* bars, WtUInt32 count)
{
	getRunner().feedRawBars(bars, count);
}
```

### 推送原始Tick数据 feed_raw_ticks
```cpp
/**
 * @brief 推送原始Tick数据
 * 
 * 将外部数据源的原始Tick数据推送到WtRtRunner中
 * 注意：此接口当前未实现，调用会记录错误日志
 * 
 * @param ticks Tick数据数组指针
 * @param count Tick数据条数
 */
void feed_raw_ticks(WTSTickStruct* ticks, WtUInt32 count)
{
	WTSLogger::error("API not implemented");  // 此接口尚未实现
}
```

### 推送复权因子数据 feed_adj_factors
```cpp
/**
 * @brief 推送复权因子数据
 * 
 * 将外部数据源的复权因子数据推送到WtRtRunner中
 * 
 * @param stdCode 标准合约代码
 * @param dates 日期数组指针（格式：YYYYMMDD）
 * @param factors 复权因子数组指针
 * @param count 数据条数
 */
void feed_adj_factors(WtString stdCode, WtUInt32* dates, double* factors, WtUInt32 count)
{
	getRunner().feedAdjFactors(stdCode, (uint32_t*)dates, factors, count);
}
```

## 系统初始化和配置

### 初始化Porter模块 init_porter
```cpp
/**
 * @brief 初始化Porter模块
 * 
 * 初始化WtPorter模块，设置日志配置和生成目录
 * 使用静态变量确保只初始化一次
 * 
 * @param logProfile 日志配置文件路径或配置内容
 * @param isFile true表示logProfile是文件路径，false表示logProfile是配置内容
 * @param genDir 生成文件目录（用于存放策略生成的文件）
 */
void init_porter(const char* logProfile, bool isFile, const char* genDir)
{
	static bool inited = false;  // 静态变量，确保只初始化一次

	if (inited)  // 如果已经初始化过，直接返回
		return;

	getRunner().init(logProfile, isFile, genDir);  // 调用WtRtRunner的初始化方法

	inited = true;  // 标记为已初始化
}
```

### 配置Porter模块 config_porter
```cpp
/**
 * @brief 配置Porter模块
 * 
 * 加载并应用配置文件，初始化交易引擎、数据管理器、交易通道、行情通道等组件
 * 
 * @param cfgfile 配置文件路径或配置内容（如果为空字符串，则使用默认配置文件"config.json"）
 * @param isFile true表示cfgfile是文件路径，false表示cfgfile是配置内容（JSON格式）
 */
void config_porter(const char* cfgfile, bool isFile)
{
	if (strlen(cfgfile) == 0)  // 如果配置文件路径为空，使用默认配置文件
		getRunner().config("config.json", true);
	else
		getRunner().config(cfgfile, isFile);  // 调用WtRtRunner的配置方法
}
```

### 运行Porter模块 run_porter
```cpp
/**
 * @brief 运行Porter模块
 * 
 * 启动交易引擎，开始接收行情和执行交易
 * 
 * @param bAsync true表示异步运行（函数立即返回），false表示同步运行（函数阻塞直到退出）
 */
void run_porter(bool bAsync)
{
	getRunner().run(bAsync);  // 调用WtRtRunner的运行方法
}
```

### 释放Porter模块 release_porter
```cpp
/**
 * @brief 释放Porter模块
 * 
 * 清理资源，停止日志系统，释放Porter模块占用的资源
 */
void release_porter()
{
	getRunner().release();  // 调用WtRtRunner的释放方法
}
```

### 写入日志 write_log
```cpp
/**
 * @brief 写入日志
 * 
 * 向日志系统写入一条日志记录
 * 
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 * @param catName 日志分类名称（可选，为空则使用默认分类）
 */
void write_log(WtUInt32 level, const char* message, const char* catName)
{
	if (strlen(catName) > 0)  // 如果指定了分类名称，使用分类日志
	{
		WTSLogger::log_raw_by_cat(catName, (WTSLogLevel)level, message);
	}
	else  // 否则使用默认日志
	{
		WTSLogger::log_raw((WTSLogLevel)level, message);
	}
}
```

### 获取版本信息 get_version
```cpp
/**
 * @brief 获取版本信息
 * 
 * 获取WonderTrader框架的版本信息字符串，包含平台、版本号、编译日期和时间
 * 使用静态变量缓存版本信息，避免重复构建字符串
 * 
 * @return 版本信息字符串（格式：平台 版本号 Build@编译日期 编译时间）
 */
const char* get_version()
{
	static std::string _ver;  // 静态变量，缓存版本信息字符串
	if (_ver.empty())  // 如果版本信息尚未构建，则构建一次
	{
		_ver = PLATFORM_NAME;  // 平台名称（X64/X86/UNIX）
		_ver += " ";
		_ver += WT_VERSION;  // 版本号
		_ver += " Build@";
		_ver += __DATE__;  // 编译日期（宏定义）
		_ver += " ";
		_ver += __TIME__;  // 编译时间（宏定义）
	}
	return _ver.c_str();  // 返回C风格字符串
}
```

## 工厂注册接口

### 注册CTA策略工厂目录 reg_cta_factories
```cpp
```

### 注册HFT策略工厂目录 reg_hft_factories
```cpp
/**
 * @brief 注册CTA策略工厂目录
 * 
 * 从指定目录加载CTA策略的动态库，注册策略工厂
 * 
 * @param factFolder 策略工厂目录路径
 * @return 是否注册成功
 */
bool reg_cta_factories(const char* factFolder)
{
	return getRunner().addCtaFactories(factFolder);  // 调用WtRtRunner的方法注册CTA策略工厂
}
```

### 注册SEL策略工厂目录 reg_sel_factories
```cpp
/**
 * @brief 注册SEL策略工厂目录
 * 
 * 从指定目录加载SEL策略的动态库，注册策略工厂
 * 
 * @param factFolder 策略工厂目录路径
 * @return 是否注册成功
 */
bool reg_sel_factories(const char* factFolder)
{
	return getRunner().addSelFactories(factFolder);  // 调用WtRtRunner的方法注册SEL策略工厂
}
```

### 注册执行器工厂目录 reg_exe_factories
```cpp
/**
 * @brief 注册执行器工厂目录
 * 
 * 从指定目录加载执行器的动态库，注册执行器工厂
 * 
 * @param factFolder 执行器工厂目录路径
 * @return 是否注册成功
 */
bool reg_exe_factories(const char* factFolder)
{
	return getRunner().addExeFactories(factFolder);  // 调用WtRtRunner的方法注册执行器工厂
}
```

## 扩展组件创建接口

### 创建扩展Parser create_ext_parser
```cpp
/**
 * @brief 创建扩展Parser
 * 
 * 在WtRtRunner中创建一个扩展Parser实例
 * 
 * @param id Parser的唯一标识符
 * @return 是否创建成功
 */
bool create_ext_parser(const char* id)
{
	return getRunner().createExtParser(id);
}
```

### 创建扩展Executer create_ext_executer
```cpp
/**
 * @brief 创建扩展Executer
 * 
 * 在WtRtRunner中创建一个扩展Executer实例
 * 
 * @param id Executer的唯一标识符
 * @return 是否创建成功
 */
bool create_ext_executer(const char* id)
{
	return getRunner().createExtExecuter(id);
}
```

### 获取原始标准代码 get_raw_stdcode
```cpp
/**
 * @brief 获取原始标准代码
 * 
 * 将标准合约代码转换为原始合约代码（去除复权、主力等后缀）
 * 
 * @param stdCode 标准合约代码（如"SHFE.rb2305.HOT"）
 * @return 原始合约代码（如"SHFE.rb2305"）
 */
const char* get_raw_stdcode(const char* stdCode)
{
	return getRunner().get_raw_stdcode(stdCode);  // 调用WtRtRunner的方法获取原始代码
}
```

## CTA策略接口

### 策略上下文管理

#### 创建CTA策略上下文 create_cta_context
```cpp
/**
 * @brief 创建CTA策略上下文
 * 
 * 创建一个新的CTA策略上下文实例
 * 
 * @param name 策略名称
 * @param slippage 滑点设置
 * @return 策略上下文句柄
 */
CtxHandler create_cta_context(const char* name, int slippage)
{
	return getRunner().createCtaContext(name, slippage);  // 调用WtRtRunner创建CTA上下文
}
```

### 交易操作

#### 开多仓 cta_enter_long
```cpp
/**
 * @brief 开多仓
 * 
 * 执行开多仓操作，买入指定数量的合约
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 开仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_enter_long(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_enter_long(stdCode, qty, userTag, limitprice, stopprice);  // 调用上下文的开多仓方法
}
```

#### 平多仓 cta_exit_long
```cpp
/**
 * @brief 平多仓
 * 
 * 执行平多仓操作，卖出指定数量的多头持仓
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 平仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_exit_long(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_exit_long(stdCode, qty, userTag, limitprice, stopprice);  // 调用上下文的平多仓方法
}
```

#### 开空仓 cta_enter_short
```cpp
/**
 * @brief 开空仓
 * 
 * 执行开空仓操作，卖出指定数量的合约
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 开仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_enter_short(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_enter_short(stdCode, qty, userTag, limitprice, stopprice);  // 调用上下文的开空仓方法
}
```

#### 平空仓 cta_exit_short
```cpp
/**
 * @brief 平空仓
 * 
 * 执行平空仓操作，买入指定数量的空头持仓
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 平仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_exit_short(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_exit_short(stdCode, qty, userTag, limitprice, stopprice);  // 调用上下文的平空仓方法
}
```

#### 设置目标持仓 cta_set_position
```cpp
/**
 * @brief 设置目标持仓
 * 
 * 设置指定合约的目标持仓数量，系统会自动计算需要开仓或平仓的数量并执行
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 目标持仓数量（正数表示多头，负数表示空头，0表示平仓）
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_set_position(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_set_position(stdCode, qty, userTag, limitprice, stopprice);  // 调用上下文的设置目标持仓方法
}
```

### 持仓查询

#### 获取持仓数量 cta_get_position
```cpp
/**
 * @brief 获取持仓数量
 * 
 * 获取指定合约的持仓数量
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param bOnlyValid true表示只返回有效持仓（已成交的），false表示返回所有持仓（包括未成交的）
 * @param openTag 开仓标签（NULL表示所有持仓，否则只返回指定标签的持仓）
 * @return 持仓数量（正数表示多头，负数表示空头，如果上下文不存在则返回0）
 */
double cta_get_position(CtxHandler cHandle, const char* stdCode, bool bOnlyValid, const char* openTag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position(stdCode, bOnlyValid, openTag);  // 调用上下文的获取持仓方法
}
```

#### 获取持仓盈亏 cta_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果上下文不存在则返回0）
 */
double cta_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用上下文的获取持仓盈亏方法
}
```

#### 获取持仓均价 cta_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果上下文不存在则返回0）
 */
double cta_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_avgpx(stdCode);  // 调用上下文的获取持仓均价方法
}
```

#### 获取明细持仓的入场时间 cta_get_detail_entertime
```cpp
/**
 * @brief 获取明细持仓的入场时间
 * 
 * 获取指定合约和开仓标签对应的持仓明细的入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签（用于区分不同的开仓批次）
 * @return 入场时间戳（格式：YYYYMMDDHHMMSS，如果上下文不存在则返回0）
 */
WtUInt64 cta_get_detail_entertime(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_entertime(stdCode, openTag);  // 调用上下文的获取明细入场时间方法
}
```

#### 获取明细持仓的成本价 cta_get_detail_cost
```cpp
/**
 * @brief 获取明细持仓的成本价
 * 
 * 获取指定合约和开仓标签对应的持仓明细的成本价
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @return 成本价（如果上下文不存在则返回0）
 */
double cta_get_detail_cost(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_cost(stdCode, openTag);  // 调用上下文的获取明细成本价方法
}
```

#### 获取明细持仓的盈亏 cta_get_detail_profit
```cpp
/**
 * @brief 获取明细持仓的盈亏
 * 
 * 获取指定合约和开仓标签对应的持仓明细的盈亏
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @param flag 盈亏类型标志（0-浮动盈亏，1-平仓盈亏）
 * @return 盈亏金额（如果上下文不存在则返回0）
 */
double cta_get_detail_profit(CtxHandler cHandle, const char* stdCode, const char* openTag, int flag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_profit(stdCode, openTag, flag);  // 调用上下文的获取明细盈亏方法
}
```

#### 获取所有持仓 cta_get_all_position
```cpp
/**
 * @brief 获取所有持仓
 * 
 * 枚举策略的所有持仓，通过回调函数返回每个持仓信息
 * 最后会调用一次回调函数，传入空字符串和isLast=true，表示枚举结束
 * 
 * @param cHandle 策略上下文句柄
 * @param cb 回调函数，用于接收持仓信息
 */
void cta_get_all_position(CtxHandler cHandle, FuncGetPositionCallback cb)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在
	{
		cb(cHandle, "", 0, true);  // 调用回调函数，传入空字符串表示无持仓，isLast=true表示结束
		return;
	}

	ctx->enum_position([cb, cHandle](const char* stdCode, double qty) {  // 使用lambda表达式枚举持仓
		cb(cHandle, stdCode, qty, false);  // 对每个持仓调用回调函数，isLast=false表示还有更多持仓
	});

	cb(cHandle, "", 0, true);  // 最后调用一次回调函数，isLast=true表示枚举结束
}
```

#### 获取首次入场时间 cta_get_first_entertime
```cpp
/**
 * @brief 获取首次入场时间
 * 
 * 获取指定合约的首次入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 首次入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果上下文不存在则返回0）
 */
WtUInt64 cta_get_first_entertime(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_first_entertime(stdCode);  // 调用上下文的获取首次入场时间方法
}
```

#### 获取首次入场时间 cta_get_first_entertime
```cpp
/**
 * @brief 获取首次入场时间
 * 
 * 获取指定合约的首次入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 首次入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果上下文不存在则返回0）
 */
WtUInt64 cta_get_first_entertime(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_first_entertime(stdCode);  // 调用上下文的获取首次入场时间方法
}
```

#### 获取最后出场时间 cta_get_last_exittime
```cpp
/**
 * @brief 获取最后入场时间
 * 
 * 获取指定合约的最后入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果上下文不存在则返回0）
 */
WtUInt64 cta_get_last_entertime(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_last_entertime(stdCode);  // 调用上下文的获取最后入场时间方法
}
```

#### 获取最后入场价格 cta_get_last_enterprice
```cpp
/**
 * @brief 获取最后入场价格
 * 
 * 获取指定合约的最后入场价格
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场价格（0表示无持仓，如果上下文不存在则返回0）
 */
double cta_get_last_enterprice(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_last_enterprice(stdCode);  // 调用上下文的获取最后入场价格方法
}
```

#### 获取最后入场标签 cta_get_last_entertag
```cpp
/**
 * @brief 获取最后入场标签
 * 
 * 获取指定合约的最后入场标签
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场标签字符串（空字符串表示无持仓，如果上下文不存在则返回NULL）
 */
WtString cta_get_last_entertag(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回NULL
		return 0;

	return ctx->stra_get_last_entertag(stdCode);  // 调用上下文的获取最后入场标签方法
}
```

### 市场数据查询

#### 获取当前价格 cta_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double cta_get_price(const char* stdCode)
{
	return getRunner().getEngine()->get_cur_price(stdCode);  // 通过引擎获取当前价格
}
```

#### 获取日线价格数据 cta_get_day_price
```cpp
/**
 * @brief 获取日线价格数据
 * 
 * 获取指定合约的日线价格数据（开盘价、最高价、最低价、收盘价等）
 * 
 * @param stdCode 标准合约代码
 * @param flag 价格类型标志（0-开盘价，1-最高价，2-最低价，3-收盘价）
 * @return 价格值
 */
double cta_get_day_price(const char* stdCode, int flag)
{
	return getRunner().getEngine()->get_day_price(stdCode, flag);  // 通过引擎获取日线价格
}
```

#### 获取K线数据 cta_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param isMain true表示主K线（用于策略计算），false表示辅助K线（仅用于查询）
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 cta_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, bool isMain, FuncGetBarsCallback cb)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = ctx->stra_get_bars(stdCode, period, barCnt, isMain);  // 从上下文获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			uint32_t blkCnt = kData->get_block_counts();  // 获取数据块数量（K线数据可能被分成多个块）
			for (uint32_t i = 0; i < blkCnt; i++)  // 遍历所有数据块
			{
				if(kData->get_block_addr(i) != NULL)  // 如果数据块地址有效
					cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == blkCnt - 1);  // 通过回调函数返回数据块
			}

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 cta_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32	cta_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = ctx->stra_get_ticks(stdCode, tickCnt);  // 从上下文获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据（一次性返回所有数据）
			tData->release();  // 释放Tick数据切片资源
			return thisCnt;  // 返回实际Tick数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

### 资金和时间查询

#### 获取资金数据 cta_get_fund_data
```cpp
/**
 * @brief 获取资金数据
 * 
 * 获取策略的资金数据（总资产、可用资金、持仓盈亏等）
 * 
 * @param cHandle 策略上下文句柄
 * @param flag 资金类型标志（0-总资产，1-可用资金，2-持仓盈亏等）
 * @return 资金数值（如果上下文不存在则返回0）
 */
double cta_get_fund_data(CtxHandler cHandle, int flag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_fund_data(flag);  // 调用上下文的获取资金数据方法
}
```

#### 获取交易日 cta_get_tdate
```cpp
/**
 * @brief 获取交易日
 * 
 * 获取当前交易日（格式：YYYYMMDD）
 * 
 * @return 交易日
 */
WtUInt32 cta_get_tdate()
{
	return getRunner().getEngine()->get_trading_date();  // 通过引擎获取交易日
}
```

#### 获取当前日期 cta_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 cta_get_date()
{
	return getRunner().getEngine()->get_date();  // 通过引擎获取当前日期
}
```

#### 获取当前时间 cta_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS）
 * 
 * @return 当前时间
 */
WtUInt32 cta_get_time()
{
	return getRunner().getEngine()->get_min_time();  // 通过引擎获取当前时间（分钟级）
}
```

### 订阅接口

#### 订阅Tick行情 cta_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void cta_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_ticks(stdCode);  // 调用上下文的订阅Tick方法
}
```

#### 订阅K线事件 cta_sub_bar_events
```cpp
/**
 * @brief 订阅K线事件
 * 
 * 订阅指定合约和周期的K线闭合事件，订阅后会在on_bar回调中收到该K线的闭合事件
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 */
void cta_sub_bar_events(CtxHandler cHandle, const char* stdCode, const char* period)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_bar_events(stdCode, period);  // 调用上下文的订阅K线事件方法
}
```

### 图表和指标接口

#### 设置图表K线 cta_set_chart_kline
```cpp
/**
 * @brief 设置图表K线
 * 
 * 为策略图表设置主K线，用于可视化展示
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 */
void cta_set_chart_kline(CtxHandler cHandle, const char* stdCode, const char* period)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->set_chart_kline(stdCode, period);  // 调用上下文的设置图表K线方法
}
```

#### 添加图表标记 cta_add_chart_mark
```cpp
/**
 * @brief 添加图表标记
 * 
 * 在策略图表上添加一个标记点（如买卖信号）
 * 
 * @param cHandle 策略上下文句柄
 * @param price 标记点的价格位置
 * @param icon 图标类型（如"buy"、"sell"等）
 * @param tag 标记标签文本
 */
void cta_add_chart_mark(CtxHandler cHandle, double price, const char* icon, const char* tag)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->add_chart_mark(price, icon, tag);  // 调用上下文的添加图表标记方法
}
```

#### 注册指标 cta_register_index
```cpp
/**
 * @brief 注册指标
 * 
 * 在策略图表上注册一个自定义指标
 * 
 * @param cHandle 策略上下文句柄
 * @param idxName 指标名称（唯一标识）
 * @param indexType 指标类型：0-主图指标（叠加在K线上），1-副图指标（独立显示）
 */
void cta_register_index(CtxHandler cHandle, const char* idxName, WtUInt32 indexType)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->register_index(idxName, indexType);  // 调用上下文的注册指标方法
}
```

#### 注册指标线 cta_register_index_line
```cpp
/**
 * @brief 注册指标线
 * 
 * 为已注册的指标添加一条数据线
 * 
 * @param cHandle 策略上下文句柄
 * @param idxName 指标名称
 * @param lineName 线条名称（唯一标识该线条）
 * @param lineType 线条类型：0-曲线，其他值可扩展
 * @return 是否注册成功（如果上下文不存在则返回false）
 */
bool cta_register_index_line(CtxHandler cHandle, const char* idxName, const char* lineName, WtUInt32 lineType)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回false
		return false;

	return ctx->register_index_line(idxName, lineName, lineType);  // 调用上下文的注册指标线方法
}
```

#### 添加指标基准线 cta_add_index_baseline
```cpp
/**
 * @brief 添加指标基准线
 * 
 * 为指标添加一条基准线（如0轴、100轴等）
 * 
 * @param cHandle 策略上下文句柄
 * @param idxName 指标名称
 * @param lineName 线条名称
 * @param val 基准线数值
 * @return 是否添加成功（如果上下文不存在则返回false）
 */
bool cta_add_index_baseline(CtxHandler cHandle, const char* idxName, const char* lineName, double val)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回false
		return false;

	return ctx->add_index_baseline(idxName, lineName, val);  // 调用上下文的添加指标基准线方法
}
```

#### 设置指标值 cta_set_index_value
```cpp
/**
 * @brief 设置指标值
 * 
 * 更新指标线的当前值（在K线闭合时调用）
 * 
 * @param cHandle 策略上下文句柄
 * @param idxName 指标名称
 * @param lineName 线条名称
 * @param val 指标值
 * @return 是否设置成功（如果上下文不存在则返回false）
 */
bool cta_set_index_value(CtxHandler cHandle, const char* idxName, const char* lineName, double val)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回false
		return false;

	return ctx->set_index_value(idxName, lineName, val);  // 调用上下文的设置指标值方法
}
```

### 日志和用户数据

#### 记录日志 cta_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void cta_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	switch (level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
		}
}
```

#### 保存用户数据 cta_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void cta_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_save_user_data(key, val);  // 调用上下文的保存用户数据方法
}
```

#### 加载用户数据 cta_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果上下文不存在则返回默认值）
 */
WtString cta_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	CtaContextPtr ctx = getRunner().getCtaContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回默认值
		return defVal;

	return ctx->stra_load_user_data(key, defVal);  // 调用上下文的加载用户数据方法
}
```

## SEL策略接口

### 策略上下文管理

#### 创建选股策略上下文 create_sel_context
```cpp
/**
 * @brief 创建选股策略上下文
 * 
 * 创建一个新的选股策略上下文实例
 * 
 * @param name 策略名称
 * @param date 策略开始日期（格式：YYYYMMDD）
 * @param time 策略开始时间（格式：HHMMSS）
 * @param period 策略执行周期（"d"-日线，"w"-周线，"m"-月线，"y"-年线，"min"-分钟线）
 * @param trdtpl 交易模板名称（默认为"CHINA"，表示中国A股市场）
 * @param session 交易时段名称（默认为"TRADING"，表示交易时段）
 * @param slippage 滑点设置（单位：最小变动价位，0表示不设置滑点）
 * @return 策略上下文句柄
 */
CtxHandler create_sel_context(const char* name, uint32_t date, uint32_t time, const char* period, const char* trdtpl/* = "CHINA"*/, const char* session/* = "TRADING"*/, int32_t slippage/* = 0*/)
{
	return getRunner().createSelContext(name, date, time, period, slippage, trdtpl, session);  // 调用WtRtRunner创建SEL上下文
}
```

### 交易操作

#### 设置目标持仓 sel_set_position
```cpp
/**
 * @brief 设置目标持仓
 * 
 * 设置指定合约的目标持仓数量，系统会自动计算需要开仓或平仓的数量并执行
 * 注意：多因子引擎中，限价和止损价格都无效，系统会使用市价执行
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param qty 目标持仓数量（正数表示多头，负数表示空头，0表示平仓）
 * @param userTag 用户标签
 */
void sel_set_position(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	//多因子引擎,限价和止价都无效
	ctx->stra_set_position(stdCode, qty, userTag);  // 调用上下文的设置目标持仓方法（多因子引擎不支持限价和止损）
}
```

### 持仓查询

#### 获取持仓数量 sel_get_position
```cpp
/**
 * @brief 获取持仓数量
 * 
 * 获取指定合约的持仓数量
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param bOnlyValid true表示只返回有效持仓（已成交的），false表示返回所有持仓
 * @param openTag 开仓标签（NULL表示所有持仓，否则只返回指定标签的持仓）
 * @return 持仓数量（正数表示多头，负数表示空头，如果上下文不存在则返回0）
 */
double sel_get_position(CtxHandler cHandle, const char* stdCode, bool bOnlyValid, const char* openTag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position(stdCode, bOnlyValid, openTag);  // 调用上下文的获取持仓方法
}
```

#### 获取持仓盈亏 sel_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果上下文不存在则返回0）
 */
double sel_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用上下文的获取持仓盈亏方法
}
```

#### 获取持仓均价 sel_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果上下文不存在则返回0）
 */
double sel_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_avgpx(stdCode);  // 调用上下文的获取持仓均价方法
}
```

#### 获取明细持仓的入场时间 sel_get_detail_entertime
```cpp
/**
 * @brief 获取明细持仓的入场时间
 * 
 * 获取指定合约和开仓标签对应的持仓明细的入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签（用于区分不同的开仓批次）
 * @return 入场时间戳（格式：YYYYMMDDHHMMSS，如果上下文不存在则返回0）
 */
WtUInt64 sel_get_detail_entertime(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_entertime(stdCode, openTag);  // 调用上下文的获取明细入场时间方法
}
```

#### 获取明细持仓的成本价 sel_get_detail_cost
```cpp
/**
 * @brief 获取明细持仓的成本价
 * 
 * 获取指定合约和开仓标签对应的持仓明细的成本价
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @return 成本价（如果上下文不存在则返回0）
 */
double sel_get_detail_cost(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_cost(stdCode, openTag);  // 调用上下文的获取明细成本价方法
}
```

#### 获取明细持仓的盈亏 sel_get_detail_profit
```cpp
/**
 * @brief 获取明细持仓的盈亏
 * 
 * 获取指定合约和开仓标签对应的持仓明细的盈亏
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @param flag 盈亏类型标志（0-浮动盈亏，1-平仓盈亏）
 * @return 盈亏金额（如果上下文不存在则返回0）
 */
double sel_get_detail_profit(CtxHandler cHandle, const char* stdCode, const char* openTag, int flag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_profit(stdCode, openTag, flag);  // 调用上下文的获取明细盈亏方法
}
```

#### 获取所有持仓 sel_get_all_position
```cpp
/**
 * @brief 获取所有持仓
 * 
 * 枚举策略的所有持仓，通过回调函数返回每个持仓信息
 * 最后会调用一次回调函数，传入空字符串和isLast=true，表示枚举结束
 * 
 * @param cHandle 策略上下文句柄
 * @param cb 回调函数，用于接收持仓信息
 */
void sel_get_all_position(CtxHandler cHandle, FuncGetPositionCallback cb)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在
	{
		cb(cHandle, "", 0, true);  // 调用回调函数，传入空字符串表示无持仓，isLast=true表示结束
		return;
	}

	ctx->enum_position([cb, cHandle](const char* stdCode, double qty) {  // 使用lambda表达式枚举持仓
		cb(cHandle, stdCode, qty, false);  // 对每个持仓调用回调函数，isLast=false表示还有更多持仓
	});

	cb(cHandle, "", 0, true);  // 最后调用一次回调函数，isLast=true表示枚举结束
}
```

#### 获取首次入场时间 sel_get_first_entertime
```cpp
/**
 * @brief 获取首次入场时间
 * 
 * 获取指定合约的首次入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 首次入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果上下文不存在则返回0）
 */
WtUInt64 sel_get_first_entertime(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_first_entertime(stdCode);  // 调用上下文的获取首次入场时间方法
}
```

#### 获取最后入场时间 sel_get_last_entertime
```cpp
/**
 * @brief 获取最后入场时间
 * 
 * 获取指定合约的最后入场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果上下文不存在则返回0）
 */
WtUInt64 sel_get_last_entertime(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_last_entertime(stdCode);  // 调用上下文的获取最后入场时间方法
}
```

#### 获取最后出场时间 sel_get_last_exittime
```cpp
/**
 * @brief 获取最后出场时间
 * 
 * 获取指定合约的最后出场时间
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后出场时间戳（格式：YYYYMMDDHHMMSS，0表示从未出场，如果上下文不存在则返回0）
 */
WtUInt64 sel_get_last_exittime(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_last_exittime(stdCode);  // 调用上下文的获取最后出场时间方法
}
```

#### 获取最后入场价格 sel_get_last_enterprice
```cpp
/**
 * @brief 获取最后入场价格
 * 
 * 获取指定合约的最后入场价格
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场价格（0表示无持仓，如果上下文不存在则返回0）
 */
double sel_get_last_enterprice(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_last_enterprice(stdCode);  // 调用上下文的获取最后入场价格方法
}
```

#### 获取最后入场标签 sel_get_last_entertag
```cpp
/**
 * @brief 获取最后入场标签
 * 
 * 获取指定合约的最后入场标签
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 最后入场标签字符串（空字符串表示无持仓，如果上下文不存在则返回NULL）
 */
WtString sel_get_last_entertag(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回NULL
		return 0;

	return ctx->stra_get_last_entertag(stdCode);  // 调用上下文的获取最后入场标签方法
}
```

### 市场数据查询

#### 获取当前价格 sel_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double sel_get_price(const char* stdCode)
{
	return getRunner().getEngine()->get_cur_price(stdCode);  // 通过引擎获取当前价格
}
```

#### 获取日线价格数据 sel_get_day_price
```cpp
/**
 * @brief 获取日线价格数据
 * 
 * 获取指定合约的日线价格数据（开盘价、最高价、最低价、收盘价等）
 * 
 * @param stdCode 标准合约代码
 * @param flag 价格类型标志（0-开盘价，1-最高价，2-最低价，3-收盘价）
 * @return 价格值
 */
double sel_get_day_price(const char* stdCode, int flag)
{
	return getRunner().getEngine()->get_day_price(stdCode, flag);  // 通过引擎获取日线价格
}
```

#### 获取K线数据 sel_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 sel_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, FuncGetBarsCallback cb)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = ctx->stra_get_bars(stdCode, period, barCnt);  // 从上下文获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			for (uint32_t i = 0; i < kData->get_block_counts(); i++)  // 遍历所有数据块
				cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  // 通过回调函数返回数据块

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 sel_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32	sel_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = ctx->stra_get_ticks(stdCode, tickCnt);  // 从上下文获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			if (thisCnt != 0)  // 如果Tick数量不为0
				cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据
			else  // 如果Tick数量为0
				cb(cHandle, stdCode, NULL, 0, true);  // 通过回调函数返回空数据（表示无数据）
			tData->release();  // 释放Tick数据切片资源
			return thisCnt;  // 返回实际Tick数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

### 资金和时间查询

#### 获取资金数据 sel_get_fund_data
```cpp
/**
 * @brief 获取资金数据
 * 
 * 获取策略的资金数据（总资产、可用资金、持仓盈亏等）
 * 
 * @param cHandle 策略上下文句柄
 * @param flag 资金类型标志（0-总资产，1-可用资金，2-持仓盈亏等）
 * @return 资金数值（如果上下文不存在则返回0）
 */
double sel_get_fund_data(CtxHandler cHandle, int flag)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_fund_data(flag);  // 调用上下文的获取资金数据方法
}
```

#### 获取交易日 sel_get_tdate
```cpp
/**
 * @brief 获取交易日
 * 
 * 获取当前交易日（格式：YYYYMMDD）
 * 
 * @return 交易日
 */
WtUInt32 sel_get_tdate()
{
	return getRunner().getEngine()->get_trading_date();  // 通过引擎获取交易日
}
```

#### 获取当前日期 sel_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 sel_get_date()
{
	return getRunner().getEngine()->get_date();  // 通过引擎获取当前日期
}
```

#### 获取当前时间 sel_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS）
 * 
 * @return 当前时间
 */
WtUInt32 sel_get_time()
{
	return getRunner().getEngine()->get_min_time();  // 通过引擎获取当前时间（分钟级）
}
```

### 订阅接口

#### 订阅Tick行情 sel_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void sel_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_ticks(stdCode);  // 调用上下文的订阅Tick方法
}
```

### 日志和用户数据

#### 记录日志 sel_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void sel_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	switch (level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
	}
}
```

#### 保存用户数据 sel_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void sel_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_save_user_data(key, val);  // 调用上下文的保存用户数据方法
}
```

#### 加载用户数据 sel_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果上下文不存在则返回默认值）
 */
WtString sel_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	SelContextPtr ctx = getRunner().getSelContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回默认值
		return defVal;

	return ctx->stra_load_user_data(key, defVal);  // 调用上下文的加载用户数据方法
}
```

## HFT策略接口

### 策略上下文管理

#### 创建HFT策略上下文 create_hft_context
```cpp
/**
 * @brief 创建HFT策略上下文
 * 
 * 创建一个新的HFT策略上下文实例
 * 
 * @param name 策略名称
 * @param trader 交易通道ID（策略绑定的交易通道）
 * @param agent true表示使用代理模式（通过执行器下单），false表示直接下单
 * @param slippage 滑点设置（单位：最小变动价位，0表示不设置滑点）
 * @return 策略上下文句柄
 */
CtxHandler create_hft_context(const char* name, const char* trader, bool agent, int32_t slippage/* = 0*/)
{
	return getRunner().createHftContext(name, trader, agent, slippage);  // 调用WtRtRunner创建HFT上下文
}
```

### 交易操作

#### 买入 hft_buy
```cpp
/**
 * @brief 买入
 * 
 * 执行买入操作，提交买入订单
 * 如果订单被拆单，会返回多个订单ID（逗号分隔）
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param price 买入价格（0表示市价）
 * @param qty 买入数量
 * @param userTag 用户标签（用于标识该笔交易）
 * @param flag 订单标志（0-普通单，其他值可扩展）
 * @return 订单ID列表（逗号分隔的字符串，如果拆单则返回多个订单ID，如果上下文不存在则返回空字符串）
 */
WtString hft_buy(CtxHandler cHandle, const char* stdCode, double price, double qty, const char* userTag, int flag)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回空字符串
		return "";

	static std::string ret;  // 静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = ctx->stra_buy(stdCode, price, qty, userTag, flag);  // 调用上下文的买入方法
	for (uint32_t localid : ids)  // 遍历所有订单ID（如果拆单则可能有多个）
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	if(ret.size() > 0)  // 如果字符串不为空
		ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}

```

#### 卖出 hft_sell
```cpp
/**
 * @brief 卖出
 * 
 * 执行卖出操作，提交卖出订单
 * 如果订单被拆单，会返回多个订单ID（逗号分隔）
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param price 卖出价格（0表示市价）
 * @param qty 卖出数量
 * @param userTag 用户标签（用于标识该笔交易）
 * @param flag 订单标志（0-普通单，其他值可扩展）
 * @return 订单ID列表（逗号分隔的字符串，如果拆单则返回多个订单ID，如果上下文不存在则返回空字符串）
 */
WtString hft_sell(CtxHandler cHandle, const char* stdCode, double price, double qty, const char* userTag, int flag)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回空字符串
		return "";

	static std::string ret;  // 静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = ctx->stra_sell(stdCode, price, qty, userTag, flag);  // 调用上下文的卖出方法
	for (uint32_t localid : ids)  // 遍历所有订单ID（如果拆单则可能有多个）
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	if (ret.size() > 0)  // 如果字符串不为空
		ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}
```

#### 撤销订单 hft_cancel
```cpp
/**
 * @brief 撤销订单
 * 
 * 撤销指定的订单
 * 
 * @param cHandle 策略上下文句柄
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @return 是否撤销成功（如果上下文不存在则返回false）
 */
bool hft_cancel(CtxHandler cHandle, WtUInt32 localid)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回false
		return false;

	return ctx->stra_cancel(localid);  // 调用上下文的撤销订单方法
}
```

#### 撤销所有订单 hft_cancel_all
```cpp
/**
 * @brief 撤销所有订单
 * 
 * 撤销指定合约和方向的所有未完成订单
 * 返回被撤销的订单ID列表（逗号分隔的字符串）
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码（NULL或空字符串表示所有合约）
 * @param isBuy true表示撤销买入订单，false表示撤销卖出订单
 * @return 被撤销的订单ID列表（逗号分隔的字符串，如果上下文不存在则返回空字符串）
 */
WtString hft_cancel_all(CtxHandler cHandle, const char* stdCode, bool isBuy)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回空字符串
		return "";

	static thread_local std::string ret;  // 线程局部静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = ctx->stra_cancel(stdCode, isBuy, DBL_MAX);  // 调用上下文的撤销所有订单方法，DBL_MAX表示撤销所有价格
	for(uint32_t localid : ids)  // 遍历所有被撤销的订单ID
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	if (ret.size() > 0)  // 如果字符串不为空
		ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}
```

### 持仓查询

#### 获取持仓数量 hft_get_position
```cpp
/**
 * @brief 获取持仓数量
 * 
 * 获取指定合约的持仓数量
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param bOnlyValid true表示只返回有效持仓（已成交的），false表示返回所有持仓（包括未成交的）
 * @return 持仓数量（正数表示多头，负数表示空头，如果上下文不存在则返回0）
 */
double hft_get_position(CtxHandler cHandle, const char* stdCode, bool bOnlyValid)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position(stdCode, bOnlyValid);  // 调用上下文的获取持仓方法
}
```

#### 获取持仓盈亏 hft_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果上下文不存在则返回0）
 */
double hft_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用上下文的获取持仓盈亏方法
}
```

#### 获取持仓均价 hft_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果上下文不存在则返回0）
 */
double hft_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_position_avgpx(stdCode);  // 调用上下文的获取持仓均价方法
}
```

#### 获取未完成订单数量 hft_get_undone
```cpp
/**
 * @brief 获取未完成订单数量
 * 
 * 获取指定合约的未完成订单数量（包括未成交和部分成交的订单）
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @return 未完成订单数量（正数表示买入未完成，负数表示卖出未完成，如果上下文不存在则返回0）
 */
double hft_get_undone(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	return ctx->stra_get_undone(stdCode);  // 调用上下文的获取未完成订单数量方法
}
```

### 市场数据查询

#### 获取当前价格 hft_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double hft_get_price(const char* stdCode)
{
	return getRunner().getEngine()->get_cur_price(stdCode);  // 通过引擎获取当前价格
}
```

#### 获取K线数据 hft_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 hft_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, FuncGetBarsCallback cb)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;

	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = ctx->stra_get_bars(stdCode, period, barCnt);  // 从上下文获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			for (uint32_t i = 0; i < kData->get_block_counts(); i++)  // 遍历所有数据块
				cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  // 通过回调函数返回数据块

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 hft_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 hft_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = ctx->stra_get_ticks(stdCode, tickCnt);  // 从上下文获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			if (thisCnt != 0)  // 如果Tick数量不为0
				cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据
			else  // 如果Tick数量为0
				cb(cHandle, stdCode, NULL, 0, true);  // 通过回调函数返回空数据（表示无数据）
			tData->release();  // 释放Tick数据切片资源
			return thisCnt;  // 返回实际Tick数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取订单队列数据 hft_get_ordque
```cpp
/**
 * @brief 获取订单队列数据
 * 
 * 获取指定合约的订单队列数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收订单队列数据
 * @return 实际返回的数据条数（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 hft_get_ordque(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetOrdQueCallback cb)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSOrdQueSlice* dataSlice = ctx->stra_get_order_queue(stdCode, itemCnt);  // 从上下文获取订单队列数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSOrdQueStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回订单队列数据
			dataSlice->release();  // 释放订单队列数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取订单明细数据 hft_get_orddtl
```cpp
/**
 * @brief 获取订单明细数据
 * 
 * 获取指定合约的订单明细数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收订单明细数据
 * @return 实际返回的数据条数（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 hft_get_orddtl(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetOrdDtlCallback cb)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSOrdDtlSlice* dataSlice = ctx->stra_get_order_detail(stdCode, itemCnt);  // 从上下文获取订单明细数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSOrdDtlStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回订单明细数据
			dataSlice->release();  // 释放订单明细数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取逐笔成交数据 hft_get_trans
```cpp
/**
 * @brief 获取逐笔成交数据
 * 
 * 获取指定合约的逐笔成交数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收逐笔成交数据
 * @return 实际返回的数据条数（如果上下文不存在或发生异常则返回0）
 */
WtUInt32 hft_get_trans(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetTransCallback cb)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTransSlice* dataSlice = ctx->stra_get_transaction(stdCode, itemCnt);  // 从上下文获取逐笔成交数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSTransStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回逐笔成交数据
			dataSlice->release();  // 释放逐笔成交数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

### 时间查询

#### 获取当前日期 hft_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 hft_get_date()
{
	return getRunner().getEngine()->get_date();  // 通过引擎获取当前日期
}
```

#### 获取当前时间 hft_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS，原始时间，包含毫秒信息）
 * 
 * @return 当前时间
 */
WtUInt32 hft_get_time()
{
	return getRunner().getEngine()->get_raw_time();  // 通过引擎获取当前原始时间（包含毫秒）
}
```

#### 获取当前秒数 hft_get_secs
```cpp
/**
 * @brief 获取当前秒数
 * 
 * 获取当前时间的秒数部分（0-59）
 * 
 * @return 当前秒数
 */
WtUInt32 hft_get_secs()
{
	return getRunner().getEngine()->get_secs();  // 通过引擎获取当前秒数
}
```

### 订阅接口

#### 订阅Tick行情 hft_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void hft_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_ticks(stdCode);  // 调用上下文的订阅Tick方法
}
```

#### 订阅订单队列 hft_sub_order_queue
```cpp
/**
 * @brief 订阅订单队列
 * 
 * 订阅指定合约的订单队列数据，订阅后会在on_order_queue回调中收到该合约的订单队列数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void hft_sub_order_queue(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_order_queues(stdCode);  // 调用上下文的订阅订单队列方法
}
```

#### 订阅订单明细 hft_sub_order_detail
```cpp
/**
 * @brief 订阅订单明细
 * 
 * 订阅指定合约的订单明细数据，订阅后会在on_order_detail回调中收到该合约的订单明细数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void hft_sub_order_detail(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_order_details(stdCode);  // 调用上下文的订阅订单明细方法
}
```

#### 订阅逐笔成交 hft_sub_transaction
```cpp
/**
 * @brief 订阅逐笔成交
 * 
 * 订阅指定合约的逐笔成交数据，订阅后会在on_transaction回调中收到该合约的逐笔成交数据
 * 
 * @param cHandle 策略上下文句柄
 * @param stdCode 标准合约代码
 */
void hft_sub_transaction(CtxHandler cHandle, const char* stdCode)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_sub_transactions(stdCode);  // 调用上下文的订阅逐笔成交方法
}
```

### 日志和用户数据

#### 记录日志 hft_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void hft_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	switch (level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
	}
}
```

#### 保存用户数据 hft_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void hft_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，直接返回
		return;

	ctx->stra_save_user_data(key, val);  // 调用上下文的保存用户数据方法
}
```

#### 加载用户数据 hft_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果上下文不存在则返回默认值）
 */
WtString hft_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	HftContextPtr ctx = getRunner().getHftContext(cHandle);  // 通过句柄获取上下文对象
	if (ctx == NULL)  // 如果上下文不存在，返回默认值
		return defVal;

	return ctx->stra_load_user_data(key, defVal);  // 调用上下文的加载用户数据方法
}
```

## 扩展Parser接口

### 推送行情数据到扩展Parser parser_push_quote
```cpp
/**
 * @brief 推送行情数据到扩展Parser
 * 
 * 将外部数据源的行情数据推送到扩展Parser中，由Parser处理后分发给策略
 * 
 * @param id Parser的唯一标识符
 * @param curTick Tick数据结构指针
 * @param uProcFlag 处理标志（用于控制数据的处理方式）
 */
void parser_push_quote(const char* id, WTSTickStruct* curTick, WtUInt32 uProcFlag)
{
	getRunner().on_ext_parser_quote(id, curTick, uProcFlag);  // 调用WtRtRunner的扩展Parser行情推送方法
}
```

# 运行时运行器 WtRtRunner.h/cpp
```cpp
class WtRtRunner : public IEngineEvtListener, public ILogHandler, public IHisDataLoader
```
负责管理整个交易系统的运行时环境。是WonderTrader框架与外部语言（如Python、C#、Java等）交互的核心桥梁，通过C接口和回调函数机制，实现了跨语言调用和事件驱动的编程模型。

## 成员
- **回调函数指针**
  - **CTA策略引擎回调函数**
    - `FuncStraInitCallback _cb_cta_init`：CTA策略初始化回调函数指针
    - `FuncSessionEvtCallback _cb_cta_sessevt`：CTA策略交易日事件回调函数指针
    - `FuncStraTickCallback _cb_cta_tick`：CTA策略Tick数据回调函数指针
    - `FuncStraCalcCallback _cb_cta_calc`：CTA策略计算回调函数指针
    - `FuncStraBarCallback _cb_cta_bar`：CTA策略K线闭合回调函数指针
    - `FuncStraCondTriggerCallback _cb_cta_cond_trigger`：CTA策略条件单触发回调函数指针
  - **SEL策略引擎回调函数**
    - `FuncStraInitCallback _cb_sel_init`：SEL策略初始化回调函数指针
    - `FuncSessionEvtCallback _cb_sel_sessevt`：SEL策略交易日事件回调函数指针
    - `FuncStraTickCallback _cb_sel_tick`：SEL策略Tick数据回调函数指针
    - `FuncStraCalcCallback _cb_sel_calc`：SEL策略计算回调函数指针
    - `FuncStraBarCallback _cb_sel_bar`：SEL策略K线闭合回调函数指针
  - **HFT策略引擎回调函数**
    - `FuncStraInitCallback _cb_hft_init`：HFT策略初始化回调函数指针
    - `FuncSessionEvtCallback _cb_hft_sessevt`：HFT策略交易日事件回调函数指针
    - `FuncStraTickCallback _cb_hft_tick`：HFT策略Tick数据回调函数指针
    - `FuncStraBarCallback _cb_hft_bar`：HFT策略K线闭合回调函数指针
    - `FuncHftChannelCallback _cb_hft_chnl`：HFT策略交易通道事件回调函数指针
    - `FuncHftOrdCallback _cb_hft_ord`：HFT策略订单回报回调函数指针
    - `FuncHftTrdCallback _cb_hft_trd`：HFT策略成交回报回调函数指针
    - `FuncHftEntrustCallback _cb_hft_entrust`：HFT策略下单结果回调函数指针
    - `FuncHftPosCallback _cb_hft_position`：HFT策略持仓变化回调函数指针
  - **HFT策略Level2数据回调函数**
    - `FuncStraOrdQueCallback _cb_hft_ordque`：HFT策略委托队列回调函数指针
    - `FuncStraOrdDtlCallback _cb_hft_orddtl`：HFT策略逐笔委托回调函数指针
    - `FuncStraTransCallback _cb_hft_trans`：HFT策略逐笔成交回调函数指针
  - **引擎事件回调函数**
    - `FuncEventCallback _cb_evt`：引擎事件回调函数指针（初始化、调度、交易日事件等）
  - **扩展Parser回调函数**
    - `FuncParserEvtCallback _cb_parser_evt`：Parser事件回调函数指针（初始化、连接、断开等）
    - `FuncParserSubCallback _cb_parser_sub`：Parser订阅回调函数指针（订阅、取消订阅合约）
  - **扩展Executer回调函数**
    - `FuncExecCmdCallback _cb_exec_cmd`：Executer命令回调函数指针（设置目标仓位）
    - `FuncExecInitCallback _cb_exec_init`：Executer初始化回调函数指针
- **核心组件**
  - **配置与管理器**
    - `WTSVariant* _config`：配置对象指针，存储加载的配置文件内容
    - `TraderAdapterMgr _traders`：交易适配器管理器，管理多个交易通道
    - `ParserAdapterMgr _parsers`：解析器适配器管理器，管理多个行情通道
    - `WtExecuterFactory _exe_factory`：执行器工厂，创建和管理执行器实例
  - **交易引擎对象**
    - `WtCtaEngine _cta_engine`：CTA策略引擎实例
    - `WtHftEngine _hft_engine`：HFT策略引擎实例
    - `WtSelEngine _sel_engine`：SEL策略引擎实例
    - `WtEngine* _engine`：当前使用的交易引擎指针（指向_cta_engine、_hft_engine或_sel_engine之一）
  - **数据管理器**
    - `WtDataStorage* _data_store`：数据存储对象指针（当前未使用，保留用于未来扩展）
    - `WtDtMgr _data_mgr`：数据管理器，管理行情数据和K线数据
  - **基础数据管理器**
    - `WTSBaseDataMgr _bd_mgr`：基础数据管理器，管理商品、合约、交易时段等基础数据
    - `WTSHotMgr _hot_mgr`：热点合约管理器，管理主力合约、次主力合约等
    - `EventNotifier _notifier`：事件通知器，用于事件推送和日志通知
  - **策略管理器**
    - `CtaStrategyMgr _cta_mgr`：CTA策略管理器，创建和管理CTA策略实例
    - `HftStrategyMgr _hft_mgr`：HFT策略管理器，创建和管理HFT策略实例
    - `SelStrategyMgr _sel_mgr`：SEL策略管理器，创建和管理SEL策略实例
    - `ActionPolicyMgr _act_policy`：开平策略管理器，管理开仓和平仓策略
- **状态标志**
  - `bool _is_hft`：是否为高频引擎标志，true表示使用HFT引擎，false表示使用CTA或SEL引擎
  - `bool _is_sel`：是否为选股引擎标志，true表示使用SEL引擎，false表示使用CTA或HFT引擎
  - `bool _to_exit`：退出标志，true表示系统需要退出，false表示系统正常运行
- **外部数据加载器**
  - `FuncLoadFnlBars _ext_fnl_bar_loader`：复权K线数据加载器函数指针，用于加载复权后的K线数据
  - `FuncLoadRawBars _ext_raw_bar_loader`：原始K线数据加载器函数指针，用于加载原始（未复权）K线数据
  - `FuncLoadAdjFactors _ext_adj_fct_loader`：复权因子加载器函数指针，用于加载复权因子数据
- **数据推送相关**
  - `void* _feed_obj`：数据接收对象指针，用于回调函数传递（由数据管理器传入）
  - `FuncReadBars _feeder_bars`：K线数据读取回调函数指针，当数据加载完成后调用此函数推送K线数据
  - `FuncReadFactors _feeder_fcts`：复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据
  - `StdUniqueMutex _feed_mtx`：数据推送互斥锁，保护数据推送相关的共享资源（线程安全）

## IHisDataLoader接口实现 - 历史数据加载接口

### 加载复权后的历史K线数据 loadFinalHisBars
当 C++ 内部的数据管理器（`WtDataMgr`）发现本地缓存或存储中缺失所需的**历史 K 线数据（特指复权后的数据）**时，它会通过此函数调用上层（如 Python 端）注册的加载器函数，从外部数据源（如 SQL 数据库、CSV 文件、第三方 API）按需加载数据。
* **调用方**：C++ 核心层的 `WtDataMgr`（数据管理器）。
* **执行方**：外部语言（Python/C#）注册的 `FuncLoadFnlBars` 回调函数。
* **数据类型**：**复权 K 线数据**（Final Bars），即经过除权除息处理后的价格数据。

采用了一种**请求-暂存-回调**的异步/同步协作模式：

1. **C++ 请求**：引擎需要某合约的 K 线，调用 `loadFinalHisBars`。
2. **保存上下文**：`WtRtRunner` 暂时保存 *是谁请求的*（`obj`）和 *数据回来给谁*（`cb`）。
3. **转发请求**：`WtRtRunner` 调用外部语言注册的函数指针 `_ext_fnl_bar_loader`。
4. **外部加载**：Python/C# 端执行数据查询逻辑。
5. **数据回送**：外部语言加载完成后，调用 `feedRawBars`，`WtRtRunner` 再利用第 2 步保存的上下文，通过 `_feeder_bars` 回调将数据塞回 C++ 引擎。

具体执行步骤：
* **线程安全锁**
  * 获取互斥锁 `StdUniqueLock lock(_feed_mtx);`。
  * **目的**：保护共享成员变量 `_feed_obj` 和 `_feeder_bars`。因为可能有多个线程同时触发数据加载请求，或者在数据回送（feed）尚未完成时又有新的请求进来，加锁防止上下文数据被覆盖或竞争。
* **检查外部加载器**
  * 检查 `_ext_fnl_bar_loader` 是否为 `NULL`。
  * **逻辑**：如果上层（Python端）没有注册过复权数据加载函数，说明不支持外部加载，直接返回 `false`。
* **保存回调上下文**
  * `_feed_obj = obj;`：保存数据请求发起者的句柄（通常是 `WtDataMgr` 内部的对象指针）。
  * `_feeder_bars = cb;`：保存接收数据的回调函数指针。
  * **作用**：当外部数据加载完毕调用 `feedRawBars` 时，需要知道这些数据该通过哪个函数（`cb`）传给哪个对象（`obj`）。
* **周期适配与转发**
  * 根据 C++ 内部的枚举类型 `period`，转换为外部接口约定的字符串格式：
    * `KP_DAY`  传入 `"d1"`
    * `KP_Minute1`  传入 `"m1"`
    * `KP_Minute5`  传入 `"m5"`
  * **异常处理**：如果传入了不支持的周期（如周线、年线），记录错误日志 `Unsupported period...` 并返回 `false`。
* **调用外部接口**
  * 执行 `return _ext_fnl_bar_loader(stdCode, "周期字符串");`。
  * **逻辑**：这行代码真正跨越了语言边界，控制权转交给 Python/C# 端的加载逻辑。
  * **返回值**：返回外部函数的执行结果（通常 `true` 表示找到了数据并开始推送，`false` 表示未找到或出错）。

```cpp
/**
 * @brief 加载复权后的历史K线数据实现（IHisDataLoader接口实现）
 * @param obj 数据接收对象指针，用于回调函数传递（由数据管理器传入）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param period K线周期（KP_DAY、KP_Minute1、KP_Minute5等）
 * @param cb 数据读取回调函数指针，当数据加载完成后调用此函数推送K线数据
 * @return 如果外部数据加载器已注册且调用成功返回true，否则返回false
 */
bool WtRtRunner::loadFinalHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb)
```

### 推送原始K线数据 feedRawBars
```cpp
/**
 * @brief 推送原始K线数据实现
 * 
 * 当外部数据加载器加载完K线数据后，调用此函数将数据推送给数据管理器。
 * 
 * @param bars K线数据数组指针，包含多个K线数据结构
 * @param count K线数据条数，表示bars数组的长度
 */
void WtRtRunner::feedRawBars(WTSBarStruct* bars, uint32_t count)
{
	if (_ext_fnl_bar_loader == NULL)
	{
		WTSLogger::error("Cannot feed bars because of no extented bar loader registered.");
		return;
	}

	_feeder_bars(_feed_obj, bars, count); // 调用K线数据读取回调函数，将数据推送给数据管理器
}
```

### 加载原始历史K线数据 loadRawHisBars
和函数 `loadFinalHisBars` 类似。
```cpp
/**
 * @brief 加载原始历史K线数据实现（IHisDataLoader接口实现）
 * 
 * 从外部数据加载器加载指定合约的原始（未复权）历史K线数据。
 * 
 * @param obj 数据接收对象指针，用于回调函数传递（由数据管理器传入）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param period K线周期（KP_DAY、KP_Minute1、KP_Minute5等）
 * @param cb 数据读取回调函数指针，当数据加载完成后调用此函数推送K线数据
 * @return 如果外部数据加载器已注册且调用成功返回true，否则返回false
 */
bool WtRtRunner::loadRawHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb)
```

### 加载所有合约的复权因子数据 loadAllAdjFactors
```cpp
/**
 * @brief 加载所有合约的复权因子数据实现（IHisDataLoader接口实现）
 * 
 * 从外部数据加载器加载所有合约的复权因子数据。
 * 
 * @param obj 数据接收对象指针，用于回调函数传递（由数据管理器传入）
 * @param cb 复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据
 * @return 如果外部数据加载器已注册且调用成功返回true，否则返回false
 */
bool WtRtRunner::loadAllAdjFactors(void* obj, FuncReadFactors cb)
{
	StdUniqueLock lock(_feed_mtx);
	if (_ext_adj_fct_loader == NULL)
		return false;

	_feed_obj = obj; // 保存数据接收对象指针，用于回调函数传递
	_feeder_fcts = cb; // 保存复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据

	return _ext_adj_fct_loader(""); // 调用外部加载器，传入空字符串表示加载所有合约的复权因子数据
}
```

### 加载指定合约的复权因子数据 loadAdjFactors
```cpp
/**
 * @brief 加载指定合约的复权因子数据实现（IHisDataLoader接口实现）
 * 
 * 从外部数据加载器加载指定合约的复权因子数据。
 * 
 * @param obj 数据接收对象指针，用于回调函数传递（由数据管理器传入）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param cb 复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据
 * @return 如果外部数据加载器已注册且调用成功返回true，否则返回false
 */
bool WtRtRunner::loadAdjFactors(void* obj, const char* stdCode, FuncReadFactors cb)
{
	StdUniqueLock lock(_feed_mtx);
	if (_ext_adj_fct_loader == NULL)
		return false;

	_feed_obj = obj; // 保存数据接收对象指针，用于回调函数传递
	_feeder_fcts = cb; // 保存复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据

	return _ext_adj_fct_loader(stdCode);  // 调用外部加载器，传入合约代码，加载指定合约的复权因子数据
}
```

### 推送复权因子数据 feedAdjFactors
```cpp
/**
 * @brief 加载指定合约的复权因子数据实现（IHisDataLoader接口实现）
 * 
 * 从外部数据加载器加载指定合约的复权因子数据。
 * 
 * @param obj 数据接收对象指针，用于回调函数传递（由数据管理器传入）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param cb 复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据
 * @return 如果外部数据加载器已注册且调用成功返回true，否则返回false
 */
bool WtRtRunner::loadAdjFactors(void* obj, const char* stdCode, FuncReadFactors cb)
{
	StdUniqueLock lock(_feed_mtx);
	if (_ext_adj_fct_loader == NULL)
		return false;

	_feed_obj = obj; // 保存数据接收对象指针，用于回调函数传递
	_feeder_fcts = cb; // 保存复权因子读取回调函数指针，当数据加载完成后调用此函数推送复权因子数据

	return _ext_adj_fct_loader(stdCode); // 调用外部加载器，传入合约代码，加载指定合约的复权因子数据
}
```

## 系统初始化和配置

### 初始化日志系统和运行环境 init

### 从配置文件加载配置并初始化各组件 config

### 运行 run
WonderTrader 运行时环境的**启动入口**。在完成所有的系统初始化（日志、数据管理器）、组件配置（行情/交易通道）以及策略加载（CTA/HFT/SEL）之后，调用此函数将正式启动整个交易系统。

核心作用是**激活各个子系统**，并根据配置决定是将控制权交还给调用者（异步模式），还是阻塞当前线程直到系统退出（同步模式）。

* **启动核心组件**
  1. **启动行情通道** (`_parsers.run()`):
     * 激活所有已注册的行情适配器（ParserAdapter）。
     * 开始连接行情源（如 CTP 行情接口），接收 Tick 数据并推送到内部系统。
  2. **启动交易通道** (`_traders.run()`):
     * 激活所有已注册的交易适配器（TraderAdapter）。
     * 开始连接交易柜台，准备接收订单指令和回报。
  3. **启动交易引擎** (`_engine->run()`):
     * 启动核心策略引擎（`WtCtaEngine`, `WtHftEngine` 或 `WtSelEngine`）。
     * 引擎开始处理行情数据、驱动策略逻辑、管理订单状态。
* **运行模式分支处理**
  * **分支 A：异步模式 (`bAsync == true`)**
    * 函数在完成上述组件启动后，**立即返回**。
    * 此时，行情接收、策略计算、交易执行都在后台线程中运行。
    * 调用者（如 Python 脚本）可以继续执行自己的逻辑（例如进入 IPython 交互界面，或运行 GUI 事件循环）。
  * **分支 B：同步模式 (`bAsync == false`)**
    * 这是默认模式，通常用于 C++ 编写的独立服务端程序。
    * **安装信号钩子 (`install_signal_hooks`)**:
      * 注册系统信号处理函数（如 `SIGINT`, `SIGTERM`）。
      * 当捕获到错误信息时，通过 `WTSLogger::error` 记录。
      * 当捕获到退出信号（如用户按 Ctrl+C）时，将成员变量 `_to_exit` 标记为 `true`，并记录日志 `Exit flag is ...`。
    * **阻塞等待**:
      * 进入 `while (!_to_exit)` 循环。
      * 每次循环调用 `std::this_thread::sleep_for` 休眠 10 毫秒，避免空转占用 CPU。
      * 直到外部信号将 `_to_exit` 置为 `true`，循环结束，函数返回。
* **异常处理**
  * 如果运行过程中发生任何标准异常或未知异常：
    * 进入 `catch (...)` 块。
    * 调用 `print_stack_trace` 获取当前的函数调用堆栈。
    * 通过回调函数将堆栈信息写入错误日志 (`WTSLogger::error`)，便于开发者排查崩溃原因。

```cpp
/**
 * @brief 运行运行时运行器实现
 * @param bAsync 是否为异步模式，true表示异步模式（不阻塞），false表示同步模式（阻塞直到系统停止）
 */
void WtRtRunner::run(bool bAsync /* = false */)
```

### 释放运行时运行器资源 release
```cpp
/**
 * @brief 释放运行时运行器资源实现
 * 
 * 停止日志系统，释放运行时运行器占用的资源。
 * 通常在程序退出前调用，确保资源正确释放。
 * 
 */
void WtRtRunner::release()
{
	WTSLogger::stop();
}
```

## 回调函数注册接口

### 注册CTA策略引擎的回调函数 registerCtaCallbacks
```cpp
/**
 * @brief 注册CTA策略引擎的回调函数实现
 * 
 * 注册CTA策略引擎的各种事件回调函数，当策略事件发生时，会调用对应的回调函数。
 * 
 * @param cbInit 策略初始化回调函数指针，当策略初始化完成时调用
 * @param cbTick Tick数据回调函数指针，当收到新的Tick数据时调用
 * @param cbCalc 策略计算回调函数指针，当策略需要计算时调用（定时计算）
 * @param cbBar K线闭合回调函数指针，当K线闭合时调用
 * @param cbSessEvt 交易日事件回调函数指针，当交易日开始或结束时调用
 * @param cbCondTrigger 条件单触发回调函数指针，当条件单触发时调用（可选，默认为NULL）
 */
void WtRtRunner::registerCtaCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, FuncStraBarCallback cbBar, 
		FuncSessionEvtCallback cbSessEvt, FuncStraCondTriggerCallback cbCondTrigger /* = NULL */)
{
	_cb_cta_init = cbInit; // 保存CTA策略初始化回调函数指针
	_cb_cta_tick = cbTick; // 保存CTA策略Tick数据回调函数指针
	_cb_cta_calc = cbCalc; // 保存CTA策略计算回调函数指针
	_cb_cta_bar = cbBar; // 保存CTA策略K线闭合回调函数指针
	_cb_cta_sessevt = cbSessEvt; // 保存CTA策略交易日事件回调函数指针
	_cb_cta_cond_trigger = cbCondTrigger; // 保存CTA策略条件单触发回调函数指针

	WTSLogger::info("Callbacks of CTA engine registration done");
}
```

### 注册SEL策略引擎的回调函数 registerSelCallbacks
```cpp
/**
 * @brief 注册SEL策略引擎的回调函数实现
 * 
 * 注册SEL（选股）策略引擎的各种事件回调函数。
 * 
 * @param cbInit 策略初始化回调函数指针，当策略初始化完成时调用
 * @param cbTick Tick数据回调函数指针，当收到新的Tick数据时调用
 * @param cbCalc 策略计算回调函数指针，当策略需要计算时调用（定时计算）
 * @param cbBar K线闭合回调函数指针，当K线闭合时调用
 * @param cbSessEvt 交易日事件回调函数指针，当交易日开始或结束时调用
 */
void WtRtRunner::registerSelCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt)
{
	_cb_sel_init = cbInit; // 保存SEL策略初始化回调函数指针
	_cb_sel_tick = cbTick; // 保存SEL策略Tick数据回调函数指针
	_cb_sel_calc = cbCalc; // 保存SEL策略计算回调函数指针
	_cb_sel_bar = cbBar; // 保存SEL策略K线闭合回调函数指针

	_cb_sel_sessevt = cbSessEvt; // 保存SEL策略交易日事件回调函数指针

	WTSLogger::info("Callbacks of SEL engine registration done");
}
```

### 注册HFT策略引擎的回调函数 registerHftCallbacks
```cpp
/**
 * @brief 注册HFT策略引擎的回调函数实现
 * 
 * 注册HFT（高频交易）策略引擎的各种事件回调函数。
 * 
 * @param cbInit 策略初始化回调函数指针，当策略初始化完成时调用
 * @param cbTick Tick数据回调函数指针，当收到新的Tick数据时调用
 * @param cbBar K线闭合回调函数指针，当K线闭合时调用
 * @param cbChnl 交易通道事件回调函数指针，当交易通道就绪或丢失时调用
 * @param cbOrd 订单回报回调函数指针，当订单状态变化时调用
 * @param cbTrd 成交回报回调函数指针，当订单成交时调用
 * @param cbEntrust 下单结果回调函数指针，当下单成功或失败时调用
 * @param cbOrdDtl 逐笔委托回调函数指针，当收到逐笔委托数据时调用
 * @param cbOrdQue 委托队列回调函数指针，当收到委托队列数据时调用
 * @param cbTrans 逐笔成交回调函数指针，当收到逐笔成交数据时调用
 * @param cbSessEvt 交易日事件回调函数指针，当交易日开始或结束时调用
 * @param cbPosition 持仓变化回调函数指针，当持仓发生变化时调用
 */
void WtRtRunner::registerHftCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraBarCallback cbBar, 
	FuncHftChannelCallback cbChnl, FuncHftOrdCallback cbOrd, FuncHftTrdCallback cbTrd, FuncHftEntrustCallback cbEntrust,
	FuncStraOrdDtlCallback cbOrdDtl, FuncStraOrdQueCallback cbOrdQue, FuncStraTransCallback cbTrans, FuncSessionEvtCallback cbSessEvt, FuncHftPosCallback cbPosition)
{
	_cb_hft_init = cbInit;  // 保存HFT策略初始化回调函数指针
	_cb_hft_tick = cbTick;  // 保存HFT策略Tick数据回调函数指针
	_cb_hft_bar = cbBar;  // 保存HFT策略K线闭合回调函数指针

	_cb_hft_chnl = cbChnl;  // 保存HFT策略交易通道事件回调函数指针
	_cb_hft_ord = cbOrd;  // 保存HFT策略订单回报回调函数指针
	_cb_hft_trd = cbTrd;  // 保存HFT策略成交回报回调函数指针
	_cb_hft_entrust = cbEntrust;  // 保存HFT策略下单结果回调函数指针

	_cb_hft_orddtl = cbOrdDtl;  // 保存HFT策略逐笔委托回调函数指针
	_cb_hft_ordque = cbOrdQue;  // 保存HFT策略委托队列回调函数指针
	_cb_hft_trans = cbTrans;  // 保存HFT策略逐笔成交回调函数指针

	_cb_hft_sessevt = cbSessEvt;  // 保存HFT策略交易日事件回调函数指针

	_cb_hft_position = cbPosition;  // 保存HFT策略持仓变化回调函数指针

	WTSLogger::info("Callbacks of HFT engine registration done");
}
```

### 注册引擎事件回调函数 registerEvtCallback
```cpp
/**
 * @brief 注册引擎事件回调函数实现
 * 
 * 注册引擎级别的事件回调函数（初始化、调度、交易日事件等），并将当前对象注册为三个引擎的事件监听器。
 * 
 * @param cbEvt 引擎事件回调函数指针，当引擎事件发生时调用
 * 
 * 注意事项：
 * - 注册后，引擎事件会通过IEngineEvtListener接口方法触发
 * - 回调函数可以为NULL，表示不处理引擎事件
 */
void WtRtRunner::registerEvtCallback(FuncEventCallback cbEvt)
{
	_cb_evt = cbEvt; // 保存引擎事件回调函数指针

	_cta_engine.regEventListener(this); // 将当前对象注册为CTA引擎的事件监听器
	_hft_engine.regEventListener(this); // 将当前对象注册为HFT引擎的事件监听器
	_sel_engine.regEventListener(this); // 将当前对象注册为SEL引擎的事件监听器
}
```

### 注册扩展Parser的回调函数 registerParserPorter
```cpp
/**
 * @brief 注册扩展Parser的回调函数实现
 * 
 * 注册扩展Parser（外部语言实现的行情解析器）的事件回调函数。
 * 
 * @param cbEvt Parser事件回调函数指针，当Parser事件发生时调用（初始化、连接、断开等）
 * @param cbSub Parser订阅回调函数指针，当需要订阅或取消订阅合约时调用
 */
void WtRtRunner::registerParserPorter(FuncParserEvtCallback cbEvt, FuncParserSubCallback cbSub)
{
	_cb_parser_evt = cbEvt; // 保存Parser事件回调函数指针
	_cb_parser_sub = cbSub; // 保存Parser订阅回调函数指针

	WTSLogger::info("Callbacks of Extented Parser registration done");
}
```

### 注册扩展Executer的回调函数 registerExecuterPorter
```cpp
/**
 * @brief 注册扩展Executer的回调函数实现
 * 
 * 注册扩展Executer（外部语言实现的执行器）的事件回调函数。
 * 
 * @param cbInit Executer初始化回调函数指针，当Executer初始化时调用
 * @param cbExec Executer命令回调函数指针，当需要执行交易命令时调用
 */
void WtRtRunner::registerExecuterPorter(FuncExecInitCallback cbInit, FuncExecCmdCallback cbExec)
{
	_cb_exec_init = cbInit; // 保存Executer初始化回调函数指针
	_cb_exec_cmd = cbExec; // 保存Executer命令回调函数指针

	WTSLogger::info("Callbacks of Extented Executer registration done");
}
```

### 注册外部数据加载器 registerExtDataLoader
```cpp
/**
 * @brief 注册外部数据加载器
 * 
 * 注册外部数据加载器函数指针，用于从自定义数据源加载历史数据。
 * 
 * @param fnlBarLoader 复权K线数据加载器函数指针，用于加载复权后的K线数据
 * @param rawBarLoader 原始K线数据加载器函数指针，用于加载原始（未复权）K线数据
 * @param fctLoader 复权因子加载器函数指针，用于加载复权因子数据
 * @param tickLoader Tick数据加载器函数指针（可选，当前未使用，默认为NULL）
 * 
 * 使用场景：
 * - 当需要从自定义数据源（如数据库、API等）加载历史数据时，注册这些加载器
 * - 数据管理器会通过IHisDataLoader接口调用这些加载器
 */
void		registerExtDataLoader(FuncLoadFnlBars fnlBarLoader, FuncLoadRawBars rawBarLoader, FuncLoadAdjFactors fctLoader, FuncLoadRawTicks tickLoader = NULL)
{
	_ext_fnl_bar_loader = fnlBarLoader; // 保存复权K线数据加载器函数指针
	_ext_raw_bar_loader = rawBarLoader; // 保存原始K线数据加载器函数指针
	_ext_adj_fct_loader = fctLoader; // 保存复权因子加载器函数指针
}
```

## 扩展组件创建接口

### 创建扩展Parser createExtParser
```cpp
/**
 * @brief 创建扩展Parser实现（外部语言实现的行情解析器）
 * 
 * 创建扩展Parser实例，并将其添加到解析器适配器管理器。
 * 
 * @param id Parser的唯一标识符，用于标识该Parser实例
 * @return 创建成功返回true，失败返回false
 */
bool WtRtRunner::createExtParser(const char* id)
{
	ParserAdapterPtr adapter(new ParserAdapter); // 创建Parser适配器智能指针实例
	ExpParser* parser = new ExpParser(id); // 创建扩展Parser实例（外部语言实现的Parser）
	adapter->initExt(id, parser, _engine, _engine->get_basedata_mgr(), _engine->get_hot_mgr());  // 初始化适配器，将扩展Parser与引擎关联
	_parsers.addAdapter(id, adapter);  // 将适配器添加到解析器适配器管理器
	WTSLogger::info("Extended parser created");
	return true;
}
```

### 创建扩展Executer createExtExecuter
```cpp
/**
 * @brief 创建扩展Executer实现（外部语言实现的执行器）
 * 
 * 创建扩展Executer实例，并将其添加到CTA引擎的执行器列表。
 * 
 * @param id Executer的唯一标识符，用于标识该Executer实例
 * @return 创建成功返回true，失败返回false
 */
bool WtRtRunner::createExtExecuter(const char* id)
{
	ExpExecuter* executer = new ExpExecuter(id); // 创建扩展执行器实例（外部语言实现的Executer）
	executer->init(); // 初始化执行器
	_cta_engine.addExecuter(ExecCmdPtr(executer)); // 将执行器添加到CTA引擎的执行器列表
	// ExecCmdPtr是执行器智能指针类型，用于管理执行器的生命周期
	WTSLogger::info("Extended Executer created");
	return true;
}
```

## 策略上下文创建和管理接口

### 创建CTA策略上下文 createCtaContext
```cpp
/**
 * @brief 创建CTA策略上下文实现
 * 
 * 创建CTA策略的扩展上下文实例，并将其添加到CTA引擎。
 * 
 * @param name 策略名称，用于标识策略
 * @param slippage 滑点设置（单位：最小变动价位），用于模拟交易时的滑点成本，默认为0
 * @return 返回策略上下文的ID，用于后续访问该上下文
 */
uint32_t WtRtRunner::createCtaContext(const char* name, int32_t slippage /* = 0 */)
{
	ExpCtaContext* ctx = new ExpCtaContext(&_cta_engine, name, slippage); // 创建CTA策略扩展上下文实例
	_cta_engine.addContext(CtaContextPtr(ctx)); // 将上下文添加到CTA引擎，引擎会自动分配上下文ID
	return ctx->id();  // 返回上下文的ID，用于后续访问该上下文
}
```

### 创建HFT策略上下文 createHftContext
```cpp
/**
 * @brief 创建HFT策略上下文实现
 * 
 * 创建HFT策略的扩展上下文实例，并将其添加到HFT引擎，同时绑定交易通道。
 * 
 * @param name 策略名称，用于标识策略
 * @param trader 交易通道ID，用于绑定交易通道
 * @param bAgent 是否为代理模式，true表示代理模式（订单直接发送到交易通道），false表示非代理模式
 * @param slippage 滑点设置（单位：最小变动价位），用于模拟交易时的滑点成本，默认为0
 * @return 返回策略上下文的ID，用于后续访问该上下文
 */
uint32_t WtRtRunner::createHftContext(const char* name, const char* trader, bool bAgent, int32_t slippage /* = 0 */)
{
	ExpHftContext* ctx = new ExpHftContext(&_hft_engine, name, bAgent, slippage); // 创建HFT策略扩展上下文实例
	_hft_engine.addContext(HftContextPtr(ctx)); // 将上下文添加到HFT引擎，引擎会自动分配上下文ID
	TraderAdapterPtr trdPtr = _traders.getAdapter(trader); // 根据交易通道ID查找交易适配器
	if(trdPtr)
	{
		ctx->setTrader(trdPtr.get()); // 将上下文绑定到交易适配器
		trdPtr->addSink(ctx);  // 将上下文添加到交易适配器的接收者列表（用于接收交易回报）
	}
	else 
	{
		WTSLogger::error("Trader {} not exists, Binding trader to HFT strategy failed", trader);  // 记录错误日志
	}
	return ctx->id();
}
```

### 创建SEL策略上下文 createSelContext
```cpp
/**
 * @brief 创建SEL策略上下文实现
 * 
 * 创建SEL（选股）策略的扩展上下文实例，并将其添加到SEL引擎。
 * 
 * @param name 策略名称，用于标识策略
 * @param date 调度日期（格式：YYYYMMDD），策略开始执行的日期
 * @param time 调度时间（格式：HHMM），策略开始执行的时间
 * @param period 调度周期字符串，可选值："d"（日）、"w"（周）、"m"（月）、"y"（年）、"min"（分钟）
 * @param slippage 滑点设置（单位：最小变动价位），用于模拟交易时的滑点成本
 * @param trdtpl 交易日模板，默认为"CHINA"（中国交易日）
 * @param session 交易时段，默认为"TRADING"（交易时段）
 * @return 返回策略上下文的ID，用于后续访问该上下文
 */
uint32_t WtRtRunner::createSelContext(const char* name, uint32_t date, uint32_t time, const char* period, int32_t slippage, const char* trdtpl /* = "CHINA" */, const char* session/* ="TRADING" */)
{
	TaskPeriodType ptype; // 调度周期类型枚举变量
	if (wt_stricmp(period, "d") == 0)  // 如果周期字符串为"d"（不区分大小写）
		ptype = TPT_Daily;  // 设置为日周期
	else if (wt_stricmp(period, "w") == 0)  // 如果周期字符串为"w"（不区分大小写）
		ptype = TPT_Weekly;  // 设置为周周期
	else if (wt_stricmp(period, "m") == 0)  // 如果周期字符串为"m"（不区分大小写）
		ptype = TPT_Monthly;  // 设置为月周期
	else if (wt_stricmp(period, "y") == 0)  // 如果周期字符串为"y"（不区分大小写）
		ptype = TPT_Yearly;  // 设置为年周期
	else if (wt_stricmp(period, "min") == 0)  // 如果周期字符串为"min"（不区分大小写）
		ptype = TPT_Minute;  // 设置为分钟周期
	else  // 如果是其他不支持的周期字符串
		ptype = TPT_None;  // 设置为无周期

	ExpSelContext* ctx = new ExpSelContext(&_sel_engine, name, slippage); // 创建SEL策略扩展上下文实例
	_sel_engine.addContext(SelContextPtr(ctx), date, time, ptype, true, trdtpl, session); // 将上下文添加到SEL引擎，并设置调度参数

	return ctx->id();
}
```

### 获取CTA策略上下文 getCtaContext
```cpp
/**
 * @brief 获取CTA策略上下文实现
 * 
 * 根据上下文ID获取CTA策略上下文指针。
 * 
 * @param id 策略上下文的ID（由createCtaContext()返回）
 * @return 返回策略上下文智能指针，如果ID不存在返回空指针
 */
CtaContextPtr WtRtRunner::getCtaContext(uint32_t id)
{
	return _cta_engine.getContext(id);  // 从CTA引擎中查找指定ID的上下文，返回智能指针
}
```

### 获取SEL策略上下文 getSelContext
```cpp
/**
 * @brief 获取SEL策略上下文实现
 * 
 * 根据上下文ID获取SEL策略上下文指针。
 * 
 * @param id 策略上下文的ID（由createSelContext()返回）
 * @return 返回策略上下文智能指针，如果ID不存在返回空指针
 */
SelContextPtr WtRtRunner::getSelContext(uint32_t id)
{
	return _sel_engine.getContext(id); // 从SEL引擎中查找指定ID的上下文，返回智能指针
}
```

### 获取HFT策略上下文 getHftContext
```cpp
/**
 * @brief 获取HFT策略上下文实现
 * 
 * 根据上下文ID获取HFT策略上下文指针。
 * 
 * @param id 策略上下文的ID（由createHftContext()返回）
 * @return 返回策略上下文智能指针，如果ID不存在返回空指针
 */
HftContextPtr WtRtRunner::getHftContext(uint32_t id)
{
	return _hft_engine.getContext(id); // 从HFT引擎中查找指定ID的上下文，返回智能指针
}
```

### 获取当前使用的交易引擎指针 getEngine
```cpp
/**
 * @brief 获取当前使用的交易引擎指针
 * 
 * 返回当前使用的交易引擎指针（CTA、HFT或SEL引擎之一）。
 * 
 * @return 返回交易引擎指针，用于访问引擎的公共接口
 */
WtEngine* getEngine(){ return _engine; }
```

### 获取原始合约代码 get_raw_stdcode
```cpp
/**
 * @brief 获取原始合约代码实现
 * 
 * 将标准合约代码转换为原始合约代码（去掉.HOT、.2ND等后缀）。
 * 
 * @param stdCode 标准合约代码，如"SHFE.rb2305.HOT"
 * @return 返回原始合约代码字符串指针，如"SHFE.rb2305"
 */
const char* WtRtRunner::get_raw_stdcode(const char* stdCode)
{
	static thread_local std::string s; // 线程局部静态变量，用于存储结果字符串（线程安全）
	s = _engine->get_rawcode(stdCode);
	return s.c_str();
}
```

## ILogHandler接口实现 - 日志处理接口

### 处理日志追加 handleLogAppend
```cpp
/**
 * @brief 处理日志追加事件实现
 * 
 * 当日志系统输出日志时，会调用此方法将日志转发给事件通知器。
 * 事件通知器可以将日志发送到外部（如邮件、短信、Webhook等）。
 * 
 * @param ll 日志级别（WTSLogLevel枚举，从100开始）
 * @param msg 日志消息内容
 * 
 * 注意事项：
 * - 此方法由日志系统回调，不应直接调用
 */
void WtRtRunner::handleLogAppend(WTSLogLevel ll, const char* msg)
{
	_notifier.notify_log(LOG_TAGS[ll-100], msg); // 将日志转发给事件通知器（通过日志级别标签和消息内容）
	// LOG_TAGS[ll-100]：将日志级别（从100开始）转换为字符串标签（数组索引从0开始）
}
```

## 扩展Parser接口

### Parser初始化事件通知 parser_init
```cpp
/**
 * @brief 扩展行情解析器初始化事件回调实现
 * 
 * 当扩展行情解析器初始化时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 */
void WtRtRunner::parser_init(const char* id)
{
	if (_cb_parser_evt) // 如果解析器事件回调函数已注册
		_cb_parser_evt(EVENT_PARSER_INIT, id); // 调用回调函数，将初始化事件转发给外部语言
}
```

### Parser连接事件通知 parser_connect
```cpp
/**
 * @brief 扩展行情解析器连接事件回调实现
 * 
 * 当扩展行情解析器连接成功时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 */
void WtRtRunner::parser_connect(const char* id)
{
	if (_cb_parser_evt)
		_cb_parser_evt(EVENT_PARSER_CONNECT, id);
}
```

### Parser释放事件通知 parser_release
```cpp
/**
 * @brief 扩展行情解析器释放事件回调实现
 * 
 * 当扩展行情解析器释放时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 */
void WtRtRunner::parser_release(const char* id)
{
	if (_cb_parser_evt)
		_cb_parser_evt(EVENT_PARSER_RELEASE, id);
}
```

### Parser断开连接事件通知 parser_disconnect
```cpp
/**
 * @brief 扩展行情解析器断开连接事件回调实现
 * 
 * 当扩展行情解析器断开连接时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 */
void WtRtRunner::parser_disconnect(const char* id)
{
	if (_cb_parser_evt)
		_cb_parser_evt(EVENT_PARSER_DISCONNECT, id);
}
```

### Parser订阅合约事件通知 parser_subscribe
```cpp
/**
 * @brief 扩展行情解析器订阅事件回调实现
 * 
 * 当扩展行情解析器订阅行情时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 * @param code 合约代码，如"SHFE.rb2305"
 */
void WtRtRunner::parser_subscribe(const char* id, const char* code)
{
	if (_cb_parser_sub)
		_cb_parser_sub(id, code, true);
}
```

### Parser取消订阅合约事件通知 parser_unsubscribe
```cpp
/**
 * @brief 扩展行情解析器取消订阅事件回调实现
 * 
 * 当扩展行情解析器取消订阅行情时，会调用此方法通知外部语言。
 * 
 * @param id 解析器ID（唯一标识符）
 * @param code 合约代码，如"SHFE.rb2305"
 */
void WtRtRunner::parser_unsubscribe(const char* id, const char* code)
{
	if (_cb_parser_sub)
		_cb_parser_sub(id, code, false);
}
```

### 处理扩展Parser推送的行情数据 on_ext_parser_quote
```cpp
/**
 * @brief 扩展行情解析器行情数据回调实现
 * 
 * 当外部语言实现的扩展行情解析器收到行情数据时，会调用此方法将行情数据推送给系统。
 * 
 * @param id 解析器ID（唯一标识符）
 * @param curTick 行情数据结构体指针（包含价格、数量等信息）
 * @param uProcFlag 处理标志（位运算，表示数据的处理方式）
 */
void WtRtRunner::on_ext_parser_quote(const char* id, WTSTickStruct* curTick, uint32_t uProcFlag)
{
	ParserAdapterPtr adapter = _parsers.getAdapter(id); // 从解析器适配器管理器中获取解析器适配器
	if (adapter) // 如果适配器存在
	{
		WTSTickData* newTick = WTSTickData::create(*curTick); // 创建行情数据对象（从结构体复制数据）
		adapter->handleQuote(newTick, uProcFlag); // 调用适配器的handleQuote方法处理行情数据
		newTick->release();
	}
	else
	{
		WTSLogger::warn("Parser {} not exists", id);
	}
}
```

## 扩展Executer接口

### Executer设置目标仓位事件通知 executer_set_position
```cpp
/**
 * @brief 扩展执行器设置目标持仓回调实现
 * 
 * 当需要设置扩展执行器的目标持仓时，会调用此方法通知外部语言。
 * 
 * @param id 执行器ID（唯一标识符）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param target 目标持仓数量（正数表示多头，负数表示空头，0表示平仓）
 */
void WtRtRunner::executer_set_position(const char* id, const char* stdCode, double target)
{
	if (_cb_exec_cmd) // 如果执行器命令回调函数已注册
		_cb_exec_cmd(id, stdCode, target); // 调用回调函数，将目标持仓命令转发给外部语言
}
```

### Executer初始化事件通知 executer_init
```cpp
/**
 * @brief 扩展执行器初始化事件回调实现
 * 
 * 当扩展执行器初始化时，会调用此方法通知外部语言。
 * 
 * @param id 执行器ID（唯一标识符）
 */
void WtRtRunner::executer_init(const char* id)
{
	if (_cb_exec_init) // 如果执行器初始化回调函数已注册
		_cb_exec_init(id); // 调用回调函数，将初始化事件转发给外部语言
}
```

## IEngineEvtListener接口实现 - 引擎事件监听接口

### 引擎初始化完成事件 on_initialize_event
```cpp
/**
 * @brief 引擎初始化完成事件（IEngineEvtListener接口实现）
 * 
 * 当交易引擎初始化完成时，引擎会调用此方法。
 * 此方法将事件转发给外部语言注册的事件回调函数。
 */
virtual void on_initialize_event() override
{
    if (_cb_evt)
        _cb_evt(EVENT_ENGINE_INIT, 0, 0);
}
```

### 引擎调度事件 on_schedule_event
```cpp
/**
 * @brief 引擎调度事件（IEngineEvtListener接口实现）
 * 
 * 当交易引擎触发调度事件时（定时触发），引擎会调用此方法。
 * 此方法将事件转发给外部语言注册的事件回调函数。
 * 
 * @param uDate 调度日期（格式：YYYYMMDD）
 * @param uTime 调度时间（格式：HHMM）
 */
virtual void on_schedule_event(uint32_t uDate, uint32_t uTime) override
{
    if (_cb_evt)
        _cb_evt(EVENT_ENGINE_SCHDL, uDate, uTime);
}
```

### 交易日事件 on_session_event
```cpp
/**
 * @brief 交易日事件（IEngineEvtListener接口实现）
 * 
 * 当交易日开始或结束时，引擎会调用此方法。
 * 此方法将事件转发给外部语言注册的事件回调函数。
 * 
 * @param uDate 交易日期（格式：YYYYMMDD）
 * @param isBegin 是否为交易日开始，true表示交易日开始，false表示交易日结束
 */
virtual void on_session_event(uint32_t uDate, bool isBegin = true) override
{
    if (_cb_evt)
        _cb_evt(isBegin ? EVENT_SESSION_BEGIN : EVENT_SESSION_END, uDate, 0);
}
```

## 策略上下文事件处理方法

### 策略初始化事件处理 ctx_on_init
```cpp
/**
 * @brief 策略初始化事件处理实现
 * 
 * 当策略上下文初始化完成时，扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_init(uint32_t id, EngineType eType/* = ET_CTA*/)
{
	switch (eType) // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_init) _cb_cta_init(id); break; // 如果是CTA引擎，调用CTA策略初始化回调函数
	case ET_HFT: if (_cb_hft_init) _cb_hft_init(id); break; // 如果是HFT引擎，调用HFT策略初始化回调函数
	case ET_SEL: if (_cb_sel_init) _cb_sel_init(id); break; // 如果是SEL引擎，调用SEL策略初始化回调函数
	default:
		break;
	}
}
```

### 策略交易日事件处理 ctx_on_session_event
```cpp
/**
 * @brief 策略交易日事件处理实现
 * 
 * 当策略的交易日开始或结束时，扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param curTDate 当前交易日期（格式：YYYYMMDD）
 * @param isBegin 是否为交易日开始，true表示交易日开始，false表示交易日结束
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_session_event(uint32_t id, uint32_t curTDate, bool isBegin /* = true */, EngineType eType /* = ET_CTA */)
{
	switch (eType) // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_sessevt) _cb_cta_sessevt(id, curTDate, isBegin); break; // 如果是CTA引擎，调用CTA策略交易日事件回调函数
	case ET_HFT: if (_cb_hft_sessevt) _cb_hft_sessevt(id, curTDate, isBegin); break; // 如果是HFT引擎，调用HFT策略交易日事件回调函数
	case ET_SEL: if (_cb_sel_sessevt) _cb_sel_sessevt(id, curTDate, isBegin); break; // 如果是SEL引擎，调用SEL策略交易日事件回调函数
	default:
		break;
	}
}
```

### 策略Tick数据事件处理 ctx_on_tick
```cpp
/**
 * @brief 策略Tick数据事件处理实现
 * 
 * 当策略收到新的Tick数据时，扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param newTick 新的Tick数据指针，包含Tick的开高低收、成交量、持仓量等数据
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_tick(uint32_t id, const char* stdCode, WTSTickData* newTick, EngineType eType /* = ET_CTA */)
{
	switch (eType)
	{
	case ET_CTA: if (_cb_cta_tick) _cb_cta_tick(id, stdCode, &newTick->getTickStruct()); break; // 如果是CTA引擎，调用CTA策略Tick数据回调函数，传入Tick数据结构指针
	case ET_HFT: if (_cb_hft_tick) _cb_hft_tick(id, stdCode, &newTick->getTickStruct()); break; // 如果是HFT引擎，调用HFT策略Tick数据回调函数，传入Tick数据结构指针
	case ET_SEL: if (_cb_sel_tick) _cb_sel_tick(id, stdCode, &newTick->getTickStruct()); break; // 如果是SEL引擎，调用SEL策略Tick数据回调函数，传入Tick数据结构指针
	default:
		break;
	}
}
```

### 策略计算事件处理 ctx_on_calc
```cpp
/**
 * @brief 策略计算事件处理实现
 * 
 * 当策略需要计算时（定时计算），扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMM）
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_calc(uint32_t id, uint32_t curDate, uint32_t curTime, EngineType eType /* = ET_CTA */)
{
	switch (eType)
	{
	case ET_CTA: if (_cb_cta_calc) _cb_cta_calc(id, curDate, curTime); break; // 如果是CTA引擎，调用CTA策略计算回调函数
	case ET_SEL: if (_cb_sel_calc) _cb_sel_calc(id, curDate, curTime); break;// 如果是SEL引擎，调用SEL策略计算回调函数
	default:
		break;
	}
}
```

### 策略K线闭合事件处理 ctx_on_bar
```cpp
/**
 * @brief 策略K线闭合事件处理实现
 * 
 * 当K线闭合时，扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param period K线周期字符串，如"m1"、"m5"、"d1"等
 * @param newBar 新的K线数据指针，包含K线的开高低收等数据
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_bar(uint32_t id, const char* stdCode, const char* period, WTSBarStruct* newBar, EngineType eType /* = ET_CTA */)
{
	switch (eType)
	{
	case ET_CTA: if (_cb_cta_bar) _cb_cta_bar(id, stdCode, period, newBar); break; // 如果是CTA引擎，调用CTA策略K线闭合回调函数
	case ET_HFT: if (_cb_hft_bar) _cb_hft_bar(id, stdCode, period, newBar); break; // 如果是HFT引擎，调用HFT策略K线闭合回调函数
	case ET_SEL: if (_cb_sel_bar) _cb_sel_bar(id, stdCode, period, newBar); break; // 如果是SEL引擎，调用SEL策略K线闭合回调函数
	default:
		break;
	}
}
```

### 策略条件单触发事件处理 ctx_on_cond_triggered
```cpp
/**
 * @brief 策略条件单触发事件处理实现
 * 
 * 当条件单触发时，扩展上下文会调用此方法。
 * 此方法根据引擎类型调用对应的回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param target 目标持仓数量，正数表示多头，负数表示空头，0表示平仓
 * @param price 触发价格，条件单触发时的价格
 * @param usertag 用户标签，用于标识条件单
 * @param eType 引擎类型（ET_CTA、ET_HFT、ET_SEL），默认为ET_CTA
 */
void WtRtRunner::ctx_on_cond_triggered(uint32_t id, const char* stdCode, double target, double price, const char* usertag, EngineType eType /* = ET_CTA */)
{
	switch (eType) // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_cond_trigger) _cb_cta_cond_trigger(id, stdCode, target, price, usertag); break; // 如果是CTA引擎，调用CTA策略条件单触发回调函数
	default: // 如果是HFT引擎、SEL引擎或其他不支持的引擎类型
		break; // 不做处理（只有CTA引擎支持条件单触发）
	}
}
```

## HFT策略专用事件处理方法

### HFT策略交易通道就绪事件处理 hft_on_channel_ready
```cpp
/**
 * @brief HFT策略交易通道就绪事件处理实现
 * 
 * 当HFT策略的交易通道就绪时，扩展上下文会调用此方法。
 * 此方法调用HFT通道事件回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param trader 交易通道ID，标识哪个交易通道就绪
 */
void WtRtRunner::hft_on_channel_ready(uint32_t cHandle, const char* trader)
{
	if (_cb_hft_chnl) // 如果HFT通道事件回调函数已注册
		_cb_hft_chnl(cHandle, trader, CHNL_EVENT_READY);
}
```

### HFT策略交易通道丢失事件处理 hft_on_channel_lost
```cpp
/**
 * @brief HFT策略交易通道丢失事件处理实现
 * 
 * 当HFT策略的交易通道丢失时，扩展上下文会调用此方法。
 * 此方法调用HFT通道事件回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param trader 交易通道ID，标识哪个交易通道丢失
 */
void WtRtRunner::hft_on_channel_lost(uint32_t cHandle, const char* trader)
{
	if (_cb_hft_chnl) // 如果HFT通道事件回调函数已注册
		_cb_hft_chnl(cHandle, trader, CHNL_EVENT_LOST); // 调用回调函数，事件类型为通道丢失
}
```

### HFT策略订单回报事件处理 hft_on_order
```cpp
/**
 * @brief HFT策略订单回报事件处理实现
 * 
 * 当HFT策略的订单状态变化时，扩展上下文会调用此方法。
 * 此方法调用HFT订单回报回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param isBuy 是否为买入订单，true表示买入，false表示卖出
 * @param totalQty 订单总数量，下单时的总数量
 * @param leftQty 订单剩余数量，未成交的数量
 * @param price 订单价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 * @param userTag 用户标签，用于标识订单（由策略在下单时传入）
 */
void WtRtRunner::hft_on_order(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled, const char* userTag)
{
	if (_cb_hft_ord) // 如果HFT订单回报回调函数已注册
		_cb_hft_ord(cHandle, localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled, userTag); // 调用回调函数，将订单回报转发给外部语言
}
```

### HFT策略成交回报事件处理 hft_on_trade
```cpp
/**
 * @brief HFT策略成交回报事件处理实现
 * 
 * 当HFT策略的订单成交时，扩展上下文会调用此方法。
 * 此方法调用HFT成交回报回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param isBuy 是否为买入成交，true表示买入，false表示卖出
 * @param vol 成交数量
 * @param price 成交价格
 * @param userTag 用户标签，用于标识订单（由策略在下单时传入）
 */
void WtRtRunner::hft_on_trade(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool isBuy, double vol, double price, const char* userTag)
{
	if (_cb_hft_trd) // 如果HFT成交回报回调函数已注册
		_cb_hft_trd(cHandle, localid, stdCode, isBuy, vol, price, userTag); // 调用回调函数，将成交回报转发给外部语言
}
```

### HFT策略下单结果事件处理 hft_on_entrust
```cpp
/**
 * @brief HFT策略下单结果事件处理实现
 * 
 * 当HFT策略的下单请求有结果时，扩展上下文会调用此方法。
 * 此方法调用HFT下单结果回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param bSuccess 是否成功，true表示下单成功，false表示下单失败
 * @param message 结果消息，如果失败则包含错误信息
 * @param userTag 用户标签，用于标识订单（由策略在下单时传入）
 */
void WtRtRunner::hft_on_entrust(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool bSuccess, const char* message, const char* userTag)
{
	if (_cb_hft_entrust) // 如果HFT下单结果回调函数已注册
		_cb_hft_entrust(cHandle, localid, stdCode, bSuccess, message, userTag); // 调用回调函数，将下单结果转发给外部语言
}
```

### HFT策略持仓变化事件处理 hft_on_position
```cpp
/**
 * @brief HFT策略持仓变化事件处理实现
 * 
 * 当HFT策略的持仓发生变化时，扩展上下文会调用此方法。
 * 此方法调用HFT持仓变化回调函数，将事件转发给外部语言。
 * 
 * @param cHandle 策略上下文的ID（HFT上下文句柄）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param isLong 是否为多头持仓，true表示多头，false表示空头
 * @param prevol 变化前持仓数量
 * @param preavail 变化前可用持仓数量
 * @param newvol 变化后持仓数量
 * @param newavail 变化后可用持仓数量
 */
void WtRtRunner::hft_on_position(uint32_t cHandle, const char* stdCode, bool isLong, double prevol, double preavail, double newvol, double newavail)
{
	if (_cb_hft_position) // 如果HFT持仓变化回调函数已注册
		_cb_hft_position(cHandle, stdCode, isLong, prevol, preavail, newvol, newavail); // 调用回调函数，将持仓变化信息转发给外部语言
}
```

### HFT策略委托队列事件处理 hft_on_order_queue
```cpp
/**
 * @brief HFT策略委托队列数据回调实现
 * 
 * 当HFT策略的委托队列数据更新时，扩展上下文会调用此方法。
 * 此方法调用HFT委托队列回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID（HFT上下文句柄）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param newOrdQue 委托队列数据对象指针
 */
void WtRtRunner::hft_on_order_queue(uint32_t id, const char* stdCode, WTSOrdQueData* newOrdQue)
{
	if (_cb_hft_ordque) // 如果HFT委托队列回调函数已注册
		_cb_hft_ordque(id, stdCode, &newOrdQue->getOrdQueStruct()); // 调用回调函数，将委托队列数据转发给外部语言
}
```

### HFT策略逐笔委托事件处理 hft_on_order_detail
```cpp
/**
 * @brief HFT策略逐笔委托数据回调实现
 * 
 * 当HFT策略的逐笔委托数据更新时，扩展上下文会调用此方法。
 * 此方法调用HFT逐笔委托回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID（HFT上下文句柄）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param newOrdDtl 逐笔委托数据对象指针
 */
void WtRtRunner::hft_on_order_detail(uint32_t id, const char* stdCode, WTSOrdDtlData* newOrdDtl)
{
	if (_cb_hft_orddtl) // 如果HFT逐笔委托回调函数已注册
		_cb_hft_orddtl(id, stdCode, &newOrdDtl->getOrdDtlStruct()); // 调用回调函数，将逐笔委托数据转发给外部语言
}
```

### HFT策略逐笔成交事件处理 hft_on_transaction
```cpp
/**
 * @brief HFT策略逐笔成交数据回调实现
 * 
 * 当HFT策略的逐笔成交数据更新时，扩展上下文会调用此方法。
 * 此方法调用HFT逐笔成交回调函数，将事件转发给外部语言。
 * 
 * @param id 策略上下文的ID（HFT上下文句柄）
 * @param stdCode 标准合约代码，如"SHFE.rb2305"
 * @param newTrans 逐笔成交数据对象指针
 */
void WtRtRunner::hft_on_transaction(uint32_t id, const char* stdCode, WTSTransData* newTrans)
{
	if (_cb_hft_trans) // 如果HFT逐笔成交回调函数已注册
		_cb_hft_trans(id, stdCode, &newTrans->getTransStruct()); // 调用回调函数，将逐笔成交数据转发给外部语言
}
```

## 工厂加载接口

### 添加执行器工厂目录 addExeFactories
```cpp
/**
 * @brief 添加执行器工厂实现
 * 
 * 从指定文件夹加载执行单元工厂动态库，注册执行单元类供后续创建执行单元实例使用。
 * 
 * @param folder 执行单元工厂文件夹路径（包含执行单元动态库的目录）
 * @return 加载成功返回true，失败返回false
 */
bool WtRtRunner::addExeFactories(const char* folder)
{
	return _exe_factory.loadFactories(folder); // 从指定文件夹加载执行单元工厂动态库
}
```

### 添加CTA策略工厂目录 addCtaFactories
```cpp
/**
 * @brief 添加CTA策略工厂实现
 * 
 * 从指定文件夹加载CTA策略工厂动态库，注册策略类供后续创建策略实例使用。
 * 
 * @param folder 策略工厂文件夹路径（包含策略动态库的目录）
 * @return 加载成功返回true，失败返回false
 */
bool WtRtRunner::addCtaFactories(const char* folder)
{
	return _cta_mgr.loadFactories(folder);
}
```

### 添加HFT策略工厂目录 addHftFactories
```cpp
/**
 * @brief 添加HFT策略工厂实现
 * 
 * 从指定文件夹加载HFT策略工厂动态库，注册策略类供后续创建策略实例使用。
 * 
 * @param folder 策略工厂文件夹路径（包含策略动态库的目录）
 * @return 加载成功返回true，失败返回false
 */
bool WtRtRunner::addHftFactories(const char* folder)
{
	return _hft_mgr.loadFactories(folder); // 从指定文件夹加载HFT策略工厂动态库
}
```

### 添加SEL策略工厂目录 addSelFactories
```cpp
/**
 * @brief 添加SEL策略工厂实现
 * 
 * 从指定文件夹加载SEL策略工厂动态库，注册策略类供后续创建策略实例使用。
 * 
 * @param folder 策略工厂文件夹路径（包含策略动态库的目录）
 * @return 加载成功返回true，失败返回false
 */
bool WtRtRunner::addSelFactories(const char* folder)
{
	return _sel_mgr.loadFactories(folder);
}
```

## 私有初始化方法

### 初始化交易通道（交易适配器）initTraders

### 初始化行情通道（解析器适配器）initParsers

### 初始化执行器 initExecuters

### 初始化数据管理器 initDataMgr

### 初始化事件通知器 initEvtNotifier

### 初始化CTA策略 initCtaStrategies

### 初始化HFT策略 initHftStrategies

### 初始化SEL策略 initSelStrategies

### 初始化开平策略管理器 initActionPolicy

### 初始化交易引擎 initEngine

# 扩展策略上下文层

## 设计思想
连接 **C++ 核心层** 与 **外部策略层（如 Python）** 的 *适配器* 或 *中转站*。即**C++ 引擎把事件推给它，它再把事件转发给你的外部代码。**

**谁调用了 `ExpCtaContext` 的方法？**
- 调用者：`WtCtaEngine`（C++ 核心 CTA 引擎）
- `ExpCtaContext` 继承自策略基类（`CtaStraBaseCtx`）。在 `WtCtaEngine`中，`ExpCtaContext` 是一个策略对象
- 当市场发生变化或时间推进时，**C++ 引擎**会自动调用 `ExpCtaContext` 中重写的虚函数。
- 例子：底层接口收到 `SHFE.rb2310` 的最新 Tick -> 推送给 `WtCtaEngine` -> 引擎遍历所有订阅了该合约的策略 -> 调用 `ExpCtaContext::on_tick_updated`。

**`ExpCtaContext` 把数据传递给了谁？（下游 / 输出端）**
- 接收者：`WtRtRunner`  外部回调函数（Python 策略接口）
- `ExpCtaContext` 自身通常不包含具体的买卖判断逻辑。它的方法体主要做两件事：
  1. 维护内部状态：调用基类 `CtaStraBaseCtx` 的方法，确保 C++ 层的资金、持仓等数据同步更新。
  2. 转发事件：调用全局单例 `WtRtRunner` 的对应接口，把事件扔出去。
- 数据流向链条：
  - `ExpCtaContext`—>`WtRtRunner::ctx_on_xxx`—>`函数指针 (Callback)`—>`Python def on_xxx(...)`

## CTA 策略扩展上下文 ExpCtaContext.h/cpp
```cpp
class ExpCtaContext : public CtaStraBaseCtx
```

### 策略初始化事件 on_init
```cpp
/**
 * @brief 策略初始化事件处理
 * 
 * 当策略上下文初始化完成时调用，先调用基类的初始化方法，然后通知外部语言策略已初始化
 */
void ExpCtaContext::on_init()
{
	CtaStraBaseCtx::on_init();
	getRunner().ctx_on_init(_context_id, ET_CTA); // 通知外部语言策略已初始化
	dump_chart_info(); // 输出图表信息（用于调试）
}
```

### 交易日开始事件 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理
 * 当新的交易日开始时调用，先调用基类方法，然后通知外部语言交易日开始
 * @param uDate 交易日（格式：YYYYMMDD）
 */
void ExpCtaContext::on_session_begin(uint32_t uDate)
{
	CtaStraBaseCtx::on_session_begin(uDate); // 调用基类方法，处理交易日开始的内部逻辑
	getRunner().ctx_on_session_event(_context_id, uDate, true, ET_CTA); // 通知外部语言交易日开始
}
```

### 交易日结束事件 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理
 * 
 * 当交易日结束时调用，先通知外部语言，然后调用基类方法
 * 
 * @param uDate 交易日（格式：YYYYMMDD）
 */
void ExpCtaContext::on_session_end(uint32_t uDate)
{
	getRunner().ctx_on_session_event(_context_id, uDate, false, ET_CTA); // 通知外部语言交易日结束
	CtaStraBaseCtx::on_session_end(uDate); // 调用基类方法，处理交易日结束的内部逻辑
}
```

### Tick更新事件 on_tick_updated
```cpp
/**
 * @brief Tick更新事件处理
 * 
 * 当订阅的合约有新的Tick数据时调用，检查是否订阅了该合约，如果是则通知外部语言
 * 
 * @param stdCode 标准合约代码
 * @param newTick 新的Tick数据指针
 */
void ExpCtaContext::on_tick_updated(const char* stdCode, WTSTickData* newTick)
{
	auto it = _tick_subs.find(stdCode); // 查找是否订阅了该合约
	if (it == _tick_subs.end()) // 如果未订阅，直接返回
		return;

	getRunner().ctx_on_tick(_context_id, stdCode, newTick, ET_CTA); // 通知外部语言Tick更新
}
```

### K线闭合事件 on_bar_close
```cpp
/**
 * @brief K线闭合事件处理
 * 当订阅的K线周期完成并生成新的K线时调用，通知外部语言K线闭合
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param newBar 新生成的K线数据结构指针
 */
void ExpCtaContext::on_bar_close(const char* stdCode, const char* period, WTSBarStruct* newBar)
{
	getRunner().ctx_on_bar(_context_id, stdCode, period, newBar, ET_CTA); // 通知外部语言K线闭合
}
```

### 策略计算事件 on_calculate
```cpp
/**
 * @brief 策略计算事件处理
 * 当引擎执行定时计算时调用，通知外部语言执行策略计算
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 */
void ExpCtaContext::on_calculate(uint32_t curDate, uint32_t curTime)
{
	getRunner().ctx_on_calc(_context_id, curDate, curTime, ET_CTA); // 通知外部语言执行策略计算
}
```

### 条件单触发事件 on_condition_triggered
```cpp
/**
 * @brief 条件单触发事件处理
 * 
 * 当设置的条件单被触发时调用，通知外部语言条件单已触发
 * 
 * @param stdCode 标准合约代码
 * @param target 目标价格
 * @param price 触发价格
 * @param usertag 用户标签
 */
void ExpCtaContext::on_condition_triggered(const char* stdCode, double target, double price, const char* usertag)
{
	getRunner().ctx_on_cond_triggered(_context_id, stdCode, target, price, usertag, ET_CTA); // 通知外部语言条件单已触发
}
```

## HFT 策略扩展上下文 ExpHftContext.h/cpp
```cpp
class ExpHftContext : public HftStraBaseCtx
```

### K线闭合事件 on_bar
```cpp
/**
 * @brief K线闭合事件处理
 * 
 * 当订阅的K线周期完成并生成新的K线时调用，先构建完整的周期字符串，然后通知外部语言
 * 
 * @param code 合约代码
 * @param period K线周期（如"m1"等）
 * @param times 周期倍数（如"m5"中的5）
 * @param newBar 新生成的K线数据结构指针
 */
void ExpHftContext::on_bar(const char* code, const char* period, uint32_t times, WTSBarStruct* newBar)
{
	if (newBar == NULL)
		return;

	thread_local static char realPeriod[8] = { 0 };
	fmtutil::format_to(realPeriod, "{}{}", period, times); // 格式化周期字符串（如"m1"+"5"->"m15"）
	getRunner().ctx_on_bar(_context_id, code, realPeriod, newBar, ET_HFT); // 通知外部语言K线闭合
	HftStraBaseCtx::on_bar(code, period, times, newBar);  // 调用基类方法，处理K线闭合的内部逻辑
}
```

### 交易通道断开事件 on_channel_lost
```cpp
/**
 * @brief 交易通道断开事件处理
 * 当交易通道断开连接时调用，先通知外部语言，然后调用基类方法
 */
void ExpHftContext::on_channel_lost()
{
	getRunner().hft_on_channel_lost(_context_id, _trader->id()); // 通知外部语言交易通道断开
	HftStraBaseCtx::on_channel_lost(); // 调用基类方法，处理通道断开的内部逻辑
}
```

### 交易通道就绪事件 on_channel_ready
```cpp
/**
 * @brief 交易通道就绪事件处理
 * 当交易通道连接成功并准备就绪时调用，先通知外部语言，然后调用基类方法
 */
void ExpHftContext::on_channel_ready()
{
	getRunner().hft_on_channel_ready(_context_id, _trader->id()); // 通知外部语言交易通道就绪
	HftStraBaseCtx::on_channel_ready(); // 调用基类方法，处理通道就绪的内部逻辑
}
```

### 委托回报事件 on_entrust
```cpp
/**
 * @brief 委托回报事件处理
 * 
 * 当委托单提交后收到回报时调用，先通知外部语言，然后调用基类方法
 * 
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param bSuccess 是否成功
 * @param message 返回消息
 */
void ExpHftContext::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message)
{
	getRunner().hft_on_entrust(_context_id, localid, stdCode, bSuccess, message, getOrderTag(localid)); // 通知外部语言委托回报
	HftStraBaseCtx::on_entrust(localid, stdCode, bSuccess, message); // 调用基类方法，处理委托回报的内部逻辑
}
```

### 策略初始化事件 on_init
```cpp
/**
 * @brief 策略初始化事件处理
 * 当策略上下文初始化完成时调用，先调用基类的初始化方法，然后通知外部语言策略已初始化
 */
void ExpHftContext::on_init()
{
	HftStraBaseCtx::on_init(); // 调用基类的初始化方法，完成基础初始化工作
	getRunner().ctx_on_init(_context_id, ET_HFT); // 通知外部语言策略已初始化
}
```

### 交易日开始事件 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理
 * 当新的交易日开始时调用，先调用基类方法，然后通知外部语言交易日开始
 * @param uTDate 交易日（格式：YYYYMMDD）
 */
void ExpHftContext::on_session_begin(uint32_t uTDate)
{
	HftStraBaseCtx::on_session_begin(uTDate); // 调用基类方法，处理交易日开始的内部逻辑
	getRunner().ctx_on_session_event(_context_id, uTDate, true, ET_HFT); // 通知外部语言交易日开始
}
```

### 交易日结束事件 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理
 * 
 * 当交易日结束时调用，先通知外部语言，然后调用基类方法
 * 
 * @param uTDate 交易日（格式：YYYYMMDD）
 */
void ExpHftContext::on_session_end(uint32_t uTDate)
{
	getRunner().ctx_on_session_event(_context_id, uTDate, false, ET_HFT); // 通知外部语言交易日结束
	HftStraBaseCtx::on_session_end(uTDate); // 调用基类方法，处理交易日结束的内部逻辑
}
```

### 订单状态变化事件 on_order
```cpp
/**
 * @brief 订单状态变化事件处理
 * 
 * 当订单状态发生变化时调用，先通知外部语言，然后调用基类方法
 * 
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy true表示买入，false表示卖出
 * @param totalQty 订单总数量
 * @param leftQty 剩余未成交数量
 * @param price 订单价格
 * @param isCanceled 是否已撤单
 */
void ExpHftContext::on_order(uint32_t localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled)
{
	getRunner().hft_on_order(_context_id, localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled, getOrderTag(localid));
	HftStraBaseCtx::on_order(localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled);
}
```

### Tick更新事件 on_tick
```cpp
/**
 * @brief Tick更新事件处理
 * 
 * 当订阅的合约有新的Tick数据时调用，先更新动态盈亏，然后检查是否订阅了该合约，
 * 如果是则通知外部语言，最后调用基类方法
 * 
 * @param code 合约代码
 * @param newTick 新的Tick数据指针
 */
void ExpHftContext::on_tick(const char* code, WTSTickData* newTick)
{
	update_dyn_profit(code, newTick); // 更新动态盈亏（基于最新价格计算持仓盈亏）
	auto it = _tick_subs.find(code);
	if (it != _tick_subs.end())
	{
		getRunner().ctx_on_tick(_context_id, code, newTick, ET_HFT);  // 通知外部语言Tick更新
	}
	HftStraBaseCtx::on_tick(code, newTick); // 调用基类方法，处理Tick更新的内部逻辑
}
```

### 订单队列更新事件 on_order_queue
```cpp
/**
 * @brief 订单队列更新事件处理
 * 
 * 当订阅的合约有新的订单队列数据时调用，通知外部语言订单队列更新
 * 
 * @param stdCode 标准合约代码
 * @param newOrdQue 新的订单队列数据指针
 */
void ExpHftContext::on_order_queue(const char* stdCode, WTSOrdQueData* newOrdQue)
{
	getRunner().hft_on_order_queue(_context_id, stdCode, newOrdQue);  // 通知外部语言订单队列更新
}
```

### 订单明细更新事件 on_order_detail
```cpp
/**
 * @brief 订单明细更新事件处理
 * 
 * 当订阅的合约有新的订单明细数据时调用，通知外部语言订单明细更新
 * 
 * @param stdCode 标准合约代码
 * @param newOrdDtl 新的订单明细数据指针
 */
void ExpHftContext::on_order_detail(const char* stdCode, WTSOrdDtlData* newOrdDtl)
{
	getRunner().hft_on_order_detail(_context_id, stdCode, newOrdDtl);  // 通知外部语言订单明细更新
}
```

### 逐笔成交更新事件 on_transaction
```cpp
/**
 * @brief 逐笔成交更新事件处理
 * 
 * 当订阅的合约有新的逐笔成交数据时调用，通知外部语言逐笔成交更新
 * 
 * @param stdCode 标准合约代码
 * @param newTrans 新的逐笔成交数据指针
 */
void ExpHftContext::on_transaction(const char* stdCode, WTSTransData* newTrans)
{
	getRunner().hft_on_transaction(_context_id, stdCode, newTrans);  // 通知外部语言逐笔成交更新
}
```

### 成交回报事件 on_trade
```cpp
/**
 * @brief 成交回报事件处理
 * 
 * 当订单有成交回报时调用，先通知外部语言，然后调用基类方法
 * 
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy true表示买入，false表示卖出
 * @param vol 成交数量
 * @param price 成交价格
 */
void ExpHftContext::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price)
{
	getRunner().hft_on_trade(_context_id, localid, stdCode, isBuy, vol, price, getOrderTag(localid));  // 通知外部语言成交回报
	HftStraBaseCtx::on_trade(localid, stdCode, isBuy, vol, price);  // 调用基类方法，处理成交回报的内部逻辑
}
```

### 持仓变化事件 on_position
```cpp
/**
 * @brief 持仓变化事件处理
 * 
 * 当持仓发生变化时调用，通知外部语言持仓变化
 * 
 * @param stdCode 标准合约代码
 * @param isLong true表示多头持仓，false表示空头持仓
 * @param prevol 变化前持仓数量
 * @param preavail 变化前可用持仓数量
 * @param newvol 变化后持仓数量
 * @param newavail 变化后可用持仓数量
 * @param tradingday 交易日（格式：YYYYMMDD）
 */
void ExpHftContext::on_position(const char* stdCode, bool isLong, double prevol, double preavail, double newvol, double newavail, uint32_t tradingday)
{
	getRunner().hft_on_position(_context_id, stdCode, isLong, prevol, preavail, newvol, newavail);
}
```

## SEL 策略扩展上下文 ExpSelContext.h/cpp
```cpp
class ExpSelContext : public SelStraBaseCtx
```

### 策略初始化事件 on_init
```cpp
/**
 * @brief 策略初始化事件处理
 * 当策略上下文初始化完成时调用，先调用基类的初始化方法，然后通知外部语言策略已初始化
 */
void ExpSelContext::on_init()
{
	SelStraBaseCtx::on_init(); // 调用基类的初始化方法，完成基础初始化工作
	getRunner().ctx_on_init(_context_id, ET_SEL); // 通知外部语言策略已初始化
}
```

### 交易日开始事件 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理
 * 
 * 当新的交易日开始时调用，先调用基类方法，然后通知外部语言交易日开始
 * 
 * @param uDate 交易日（格式：YYYYMMDD）
 */
void ExpSelContext::on_session_begin(uint32_t uDate)
{
	SelStraBaseCtx::on_session_begin(uDate); // 调用基类方法，处理交易日开始的内部逻辑
	getRunner().ctx_on_session_event(_context_id, uDate, true, ET_SEL); // 通知外部语言交易日开始
}
```

### 交易日结束事件 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理
 * 
 * 当交易日结束时调用，先通知外部语言，然后调用基类方法
 * 
 * @param uDate 交易日（格式：YYYYMMDD）
 */
void ExpSelContext::on_session_end(uint32_t uDate)
{
	getRunner().ctx_on_session_event(_context_id, uDate, false, ET_SEL); // 通知外部语言交易日结束
	SelStraBaseCtx::on_session_end(uDate); // 调用基类方法，处理交易日结束的内部逻辑
}
```

### 策略调度事件 on_strategy_schedule
```cpp
/**
 * @brief 策略调度事件处理
 * 当策略到达调度时间时调用，通知外部语言执行策略计算
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 */
void ExpSelContext::on_strategy_schedule(uint32_t curDate, uint32_t curTime)
{
	getRunner().ctx_on_calc(_context_id, curDate, curTime, ET_SEL); // 通知外部语言执行策略计算
}
```

### K线闭合事件 on_bar_close
```cpp
/**
 * @brief K线闭合事件处理
 * 
 * 当订阅的K线周期完成并生成新的K线时调用，通知外部语言K线闭合
 * 
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param newBar 新生成的K线数据结构指针
 */
void ExpSelContext::on_bar_close(const char* stdCode, const char* period, WTSBarStruct* newBar)
{
	getRunner().ctx_on_bar(_context_id, stdCode, period, newBar, ET_SEL);
}
```

### Tick更新事件 on_tick_updated
```cpp
/**
 * @brief Tick更新事件处理
 * 
 * 当订阅的合约有新的Tick数据时调用，检查是否订阅了该合约，如果是则通知外部语言
 * 
 * @param stdCode 标准合约代码
 * @param newTick 新的Tick数据指针
 */
void ExpSelContext::on_tick_updated(const char* stdCode, WTSTickData* newTick)
{
	auto it = _tick_subs.find(stdCode);  // 查找是否订阅了该合约
	if (it == _tick_subs.end())
		return;
	getRunner().ctx_on_tick(_context_id, stdCode, newTick, ET_SEL); // 通知外部语言Tick更新
}
```

# 扩展组件层

## 扩展行情解析器 ExpParser.h/cpp
```cpp
class ExpParser : public IParserApi
```
继承自IParserApi接口，用于实现扩展的行情解析器。是外部语言（如Python）实现Parser的桥梁。

### 成员
- `std::string _id`：Parser的唯一标识符
- `IParserSpi* m_sink`：事件监听器（用于分发行情数据）
- `IBaseDataMgr* m_pBaseDataMgr`：基础数据管理器指针

### 初始化Parser init
```cpp
/**
 * @brief 初始化Parser
 * 
 * 调用WtRtRunner的parser_init方法，触发外部语言实现的初始化回调
 * 
 * @param config 配置信息（当前未使用）
 * @return 总是返回true
 */
bool ExpParser::init(WTSVariant* config)
{
	getRunner().parser_init(_id.c_str());  // 通知外部语言Parser初始化事件
	return true;
}
```

### 释放Parser release
```cpp
/**
 * @brief 释放Parser
 * 
 * 调用WtRtRunner的parser_release方法，触发外部语言实现的释放回调
 */
void ExpParser::release()
{
	getRunner().parser_release(_id.c_str());  // 通知外部语言Parser释放事件
}
```

### 连接数据源 connect
```cpp
/**
 * @brief 连接数据源
 * 
 * 调用WtRtRunner的parser_connect方法，触发外部语言实现的连接回调
 * 
 * @return 总是返回true（实际连接状态由外部语言控制）
 */
bool ExpParser::connect()
{
	getRunner().parser_connect(_id.c_str());  // 通知外部语言Parser连接事件
	return true;
}
```

### 断开数据源连接 disconnect
```cpp
/**
 * @brief 断开数据源连接
 * 
 * 调用WtRtRunner的parser_disconnect方法，触发外部语言实现的断开回调
 * 
 * @return 总是返回true
 */
bool ExpParser::disconnect()
{
	getRunner().parser_disconnect(_id.c_str());  // 通知外部语言Parser断开连接事件
	return true;
}
```

### 检查连接状态 isConnected
```cpp
/**
 * @brief 检查连接状态
 * 
 * 检查Parser是否已连接到数据源
 * 
 * @return 连接状态（扩展Parser始终返回true，由外部语言控制实际连接状态）
 */
virtual bool isConnected() override { return true; }
```

### 订阅合约 subscribe
```cpp
/**
 * @brief 订阅合约
 * 
 * 遍历合约代码集合，对每个合约调用WtRtRunner的parser_subscribe方法，
 * 触发外部语言实现的订阅回调
 * 
 * @param setCodes 合约代码集合
 */
void ExpParser::subscribe(const CodeSet& setCodes)
{
	for(const auto& code : setCodes)  // 遍历所有合约代码
		getRunner().parser_subscribe(_id.c_str(), code.c_str());  // 通知外部语言订阅该合约
}
```

### 取消订阅合约 unsubscribe
```cpp
/**
 * @brief 取消订阅合约
 * 
 * 遍历合约代码集合，对每个合约调用WtRtRunner的parser_unsubscribe方法，
 * 触发外部语言实现的取消订阅回调
 * 
 * @param setCodes 合约代码集合
 */
void ExpParser::unsubscribe(const CodeSet& setCodes)
{
	for (const auto& code : setCodes)  // 遍历所有合约代码
		getRunner().parser_unsubscribe(_id.c_str(), code.c_str());  // 通知外部语言取消订阅该合约
}
```

### 注册事件监听器 registerSpi
```cpp
/**
 * @brief 注册事件监听器
 * 
 * 保存事件监听器指针，并获取基础数据管理器指针
 * 
 * @param listener 事件监听器指针（用于接收和分发行情数据）
 */
void ExpParser::registerSpi(IParserSpi* listener)
{
	m_sink = listener;  // 保存事件监听器指针

	if (m_sink)  // 如果监听器有效，获取基础数据管理器
		m_pBaseDataMgr = m_sink->getBaseDataMgr();  // 获取基础数据管理器，用于查询合约信息等
}
```

## 扩展执行器 ExpExecuter.h/cpp

### 初始化执行器 init
```cpp
/**
 * @brief 初始化执行器
 * 
 * 调用WtRtRunner的executer_init方法，触发外部语言实现的初始化回调
 */
void ExpExecuter::init()
{
	getRunner().executer_init(name());  // 通知外部语言执行器初始化事件
}
```

### 设置目标持仓 set_position
```cpp
/**
 * @brief 设置目标持仓
 * 
 * 遍历目标持仓映射表，对每个合约调用WtRtRunner的executer_set_position方法，
 * 触发外部语言实现的命令回调
 * 
 * @param targets 目标持仓映射表（合约代码->目标持仓数量）
 */
void ExpExecuter::set_position(const wt_hashmap<std::string, double>& targets)
{
	for(auto& v : targets)  // 遍历所有目标持仓
	{
		getRunner().executer_set_position(name(), v.first.c_str(), v.second);  // 通知外部语言调整该合约的持仓
	}
}
```

### 持仓变化事件 on_position_changed
```cpp
/**
 * @brief 持仓变化事件处理
 * 
 * 当策略持仓发生变化时调用，通知外部语言调整持仓
 * 
 * @param stdCode 标准合约代码
 * @param targetPos 目标持仓数量
 */
void ExpExecuter::on_position_changed(const char* stdCode, double targetPos)
{
	getRunner().executer_set_position(name(), stdCode, targetPos);  // 通知外部语言调整持仓
}
```